## 0. Khởi tạo môi trường


In [1]:
import os
import sys
import re
import math
import sqlite3
import shutil
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import orjson
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedShuffleSplit

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

for folder in ['data/df', 'data/dataset', 'docs/main_paper/chart_img']:
    Path(folder).mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]} | PyArrow: {pa.__version__} | NumPy: {np.__version__} | Pandas: {pd.__version__}")
print("Khởi tạo môi trường thành công!")

Python: 3.11.18 | PyArrow: 17.0.0 | NumPy: 1.26.4 | Pandas: 3.0.6
Khởi tạo môi trường thành công!


## 1. Import dataset


In [2]:
json_candidates = [
    Path("data/dataset/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap_dts/TripAdvisor_EN.json"),
    Path("/kaggle/input/dap-dts/TripAdvisor_EN.json"),
    Path("TripAdvisor_EN.json")
]
dataset_path = next((p for p in json_candidates if p.exists()), None)
if dataset_path is None and Path("/kaggle/input").exists():
    found_jsons = list(Path("/kaggle/input").rglob("TripAdvisor_EN.json"))
    if found_jsons:
        dataset_path = found_jsons[0]
if dataset_path is None:
    dataset_path = Path("data/dataset/TripAdvisor_EN.json")

with open(dataset_path, "rb") as f:
    raw_json = orjson.loads(f.read())
print(f"Đã đọc {len(raw_json):,} bản ghi bằng orjson.")

reviews_list = []
annotations_list = []

for idx, item in enumerate(raw_json, 1):
    review_id = idx
    raw_item_id = item.get("id")
    data_block = item.get("data", {})
    raw_text = unicodedata.normalize("NFC", data_block.get("text", "").strip())
    meta = data_block.get("meta_info", {})

    pos_c, neg_c, neu_c = 0, 0, 0
    ann_container = item.get("annotations", [])
    if ann_container:
        results = ann_container[0].get("result", [])
        entities_dict = {}
        sentiments_dict = {}

        for res in results:
            res_id = res.get("id")
            fn = res.get("from_name")
            val = res.get("value", {})
            lbls = val.get("labels", [])
            if not lbls:
                continue
            if fn == "entities":
                entities_dict[res_id] = (lbls[0], int(val.get("start", 0)), int(val.get("end", 0)), val.get("text", ""))
            elif fn == "entity_sentiment":
                sentiments_dict[res_id] = lbls[0]

        for res_id, (aspect, start_c, end_c, span_txt) in entities_dict.items():
            sentiment = sentiments_dict.get(res_id, "Neutral")
            if sentiment == "Positive":
                pos_c += 1
            elif sentiment == "Negative":
                neg_c += 1
            else:
                neu_c += 1
            annotations_list.append({
                "annotation_id": f"{review_id}_{res_id}",
                "review_id": review_id,
                "aspect": aspect,
                "sentiment": sentiment,
                "start_char": start_c,
                "end_char": end_c,
                "span_text": span_txt.strip()
            })

    if pos_c > neg_c and pos_c >= neu_c:
        overall_sentiment, sentiment_id = "Positive", 2
    elif neg_c > pos_c and neg_c >= neu_c:
        overall_sentiment, sentiment_id = "Negative", 0
    elif pos_c > neg_c:
        overall_sentiment, sentiment_id = "Positive", 2
    elif neg_c > pos_c:
        overall_sentiment, sentiment_id = "Negative", 0
    else:
        overall_sentiment, sentiment_id = "Neutral", 1

    reviews_list.append({
        "review_id": review_id,
        "task_id": raw_item_id,
        "ref_id": meta.get("ref_id", f"TripAdvisor_En_{review_id}"),
        "hotel_name": meta.get("hotel name", "").strip(),
        "hotel_address": meta.get("hotel address", "").strip(),
        "location": meta.get("location", "").strip(),
        "star": meta.get("star", "").strip(),
        "overall_sentiment": overall_sentiment,
        "sentiment_id": sentiment_id,
        "date": meta.get("date", "").strip(),
        "year": int(meta.get("year", 0)) if str(meta.get("year", "")).isdigit() else None,
        "month": int(meta.get("month", 0)) if str(meta.get("month", "")).isdigit() else None,
        "review_text": raw_text,
        "char_length": len(raw_text),
        "word_count": len(raw_text.split())
    })

df_reviews = pd.DataFrame(reviews_list)
df_annotations = pd.DataFrame(annotations_list)

print(f"Tổng số reviews: {len(df_reviews):,} | Tổng số aspect annotations: {len(df_annotations):,}")

Đã đọc 9,990 bản ghi bằng orjson.
Tổng số reviews: 9,990 | Tổng số aspect annotations: 54,326


## 2. Dataframe và SQL Queries


### 2.1 Khởi tạo Dataframe


In [3]:
os.makedirs("data/df", exist_ok=True)
db_path = "data/df/chatbot.db"

if Path("chatbot.db").exists():
    Path("chatbot.db").unlink()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS reviews;")
cursor.execute("DROP TABLE IF EXISTS aspect_annotations;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS reviews (
    review_id INTEGER PRIMARY KEY,
    task_id INTEGER,
    ref_id TEXT,
    hotel_name TEXT,
    hotel_address TEXT,
    location TEXT,
    star TEXT,
    overall_sentiment TEXT,
    sentiment_id INTEGER,
    date TEXT,
    year INTEGER,
    month INTEGER,
    review_text TEXT,
    char_length INTEGER,
    word_count INTEGER
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS aspect_annotations (
    annotation_id TEXT PRIMARY KEY,
    review_id INTEGER,
    aspect TEXT,
    sentiment TEXT,
    start_char INTEGER,
    end_char INTEGER,
    span_text TEXT,
    FOREIGN KEY (review_id) REFERENCES reviews(review_id)
);
""")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_sentiment ON reviews(sentiment_id, overall_sentiment);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_location ON reviews(location);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_aspect_sentiment ON aspect_annotations(aspect, sentiment);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ann_review_id ON aspect_annotations(review_id);")

df_reviews.to_sql("reviews", conn, if_exists="replace", index=False)
df_annotations.to_sql("aspect_annotations", conn, if_exists="replace", index=False)
conn.commit()

r_cnt = cursor.execute("SELECT COUNT(*) FROM reviews").fetchone()[0]
a_cnt = cursor.execute("SELECT COUNT(*) FROM aspect_annotations").fetchone()[0]
print(f"Khởi tạo CSDL tại '{db_path}' thành công! Số reviews: {r_cnt:,} | Số annotations: {a_cnt:,}")

Khởi tạo CSDL tại 'data/df/chatbot.db' thành công! Số reviews: 9,990 | Số annotations: 54,326


### 2.2 SQL Queries cho các Research Questions (RQs)


In [4]:
query_1 = """
SELECT
    sentiment_id,
    overall_sentiment,
    COUNT(review_id) AS total_reviews,
    ROUND(100.0 * COUNT(review_id) / (SELECT COUNT(*) FROM reviews), 2) AS pct_share
FROM reviews
GROUP BY sentiment_id, overall_sentiment
ORDER BY sentiment_id ASC;
"""
df_q1 = pd.read_sql_query(query_1, conn)
print("Kết quả Query 1 - RQ1:")
display(df_q1)

Kết quả Query 1 - RQ1:


,sentiment_id,overall_sentiment,total_reviews,pct_share
0,0,Negative,668,6.69
1,1,Neutral,314,3.14
2,2,Positive,9008,90.17


In [5]:
query_2 = """
SELECT
    r.sentiment_id,
    r.overall_sentiment,
    COUNT(a.annotation_id) AS total_rationales,
    ROUND(AVG(a.end_char - a.start_char), 2) AS avg_rationale_chars,
    ROUND(AVG(CAST(a.end_char - a.start_char AS REAL) / r.char_length) * 100, 2) AS avg_evidence_coverage_pct
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE r.char_length > 0
GROUP BY r.sentiment_id, r.overall_sentiment
ORDER BY r.sentiment_id ASC;
"""
df_q2 = pd.read_sql_query(query_2, conn)
print("Kết quả Query 2 - RQ1:")
display(df_q2)

Kết quả Query 2 - RQ1:


,sentiment_id,overall_sentiment,total_rationales,avg_rationale_chars,avg_evidence_coverage_pct
0,0,Negative,3506,72.29,14.00
1,1,Neutral,910,72.05,15.02
2,2,Positive,49910,62.81,14.53


In [6]:
query_3 = """
SELECT
    a1.aspect AS aspect_x,
    a2.aspect AS aspect_y,
    COUNT(DISTINCT a1.review_id) AS co_occurrence_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect < a2.aspect
GROUP BY aspect_x, aspect_y
ORDER BY co_occurrence_count DESC;
"""
df_q3 = pd.read_sql_query(query_3, conn)
print("Kết quả Query 3 - RQ2:")
display(df_q3)

Kết quả Query 3 - RQ2:


,aspect_x,aspect_y,co_occurrence_count
0,Facility,Service,6932
1,Experience,Service,5512
2,Amenity,Service,5224
3,Amenity,Facility,4496
4,Experience,Facility,4483
5,Loyalty,Service,3717
6,Amenity,Experience,3364
7,Facility,Loyalty,2873
8,Experience,Loyalty,2414
9,Amenity,Loyalty,2181


In [7]:
query_4 = """
SELECT
    a1.aspect AS aspect_pos,
    a2.aspect AS aspect_neg,
    COUNT(DISTINCT a1.review_id) AS conflicting_reviews_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.aspect != a2.aspect
WHERE a1.sentiment = 'Positive' AND a2.sentiment = 'Negative'
GROUP BY aspect_pos, aspect_neg
ORDER BY conflicting_reviews_count DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(query_4, conn)
print("Kết quả Query 4 - RQ2:")
display(df_q4)

Kết quả Query 4 - RQ2:


,aspect_pos,aspect_neg,conflicting_reviews_count
0,Service,Facility,640
1,Amenity,Facility,480
2,Facility,Service,384
3,Service,Experience,359
4,Experience,Facility,337
5,Facility,Experience,316
6,Amenity,Service,281
7,Amenity,Experience,246
8,Experience,Service,203
9,Service,Amenity,187


In [8]:
query_5 = """
SELECT
    a1.aspect AS aspect_1,
    a2.aspect AS aspect_2,
    COUNT(*) AS collision_count
FROM aspect_annotations a1
JOIN aspect_annotations a2 ON a1.review_id = a2.review_id AND a1.annotation_id < a2.annotation_id
WHERE a1.aspect != a2.aspect
  AND NOT (a1.end_char <= a2.start_char OR a1.start_char >= a2.end_char)
GROUP BY aspect_1, aspect_2
ORDER BY collision_count DESC;
"""
df_q5 = pd.read_sql_query(query_5, conn)
print("Kết quả Query 5 - RQ3:")
display(df_q5)

Kết quả Query 5 - RQ3:


,aspect_1,aspect_2,collision_count
0,Facility,Service,2
1,Amenity,Facility,1
2,Experience,Amenity,1
3,Facility,Experience,1
4,Loyalty,Amenity,1
5,Service,Experience,1
6,Service,Loyalty,1


In [9]:
query_6 = """
SELECT
    r.sentiment_id,
    r.overall_sentiment,
    a.aspect,
    COUNT(DISTINCT r.review_id) AS review_count,
    SUM(CASE WHEN a.sentiment = 'Positive' THEN 1 ELSE 0 END) AS pos_cnt,
    SUM(CASE WHEN a.sentiment = 'Neutral' THEN 1 ELSE 0 END) AS neu_cnt,
    SUM(CASE WHEN a.sentiment = 'Negative' THEN 1 ELSE 0 END) AS neg_cnt,
    ROUND(CAST(COUNT(r.review_id) AS REAL) / (SELECT COUNT(*) FROM reviews), 4) AS class_prior
FROM reviews r
JOIN aspect_annotations a ON r.review_id = a.review_id
WHERE a.aspect IN ('Service', 'Facility', 'Experience')
GROUP BY r.sentiment_id, r.overall_sentiment, a.aspect
ORDER BY a.aspect, r.sentiment_id ASC;
"""
df_q6 = pd.read_sql_query(query_6, conn)
print("Kết quả Query 6 - RQ3:")
display(df_q6)

Kết quả Query 6 - RQ3:


,sentiment_id,overall_sentiment,aspect,review_count,pos_cnt,neu_cnt,neg_cnt,class_prior
0,0,Negative,Experience,385,58,58,494,0.0611
1,1,Neutral,Experience,100,46,32,72,0.0150
2,2,Positive,Experience,5544,7627,420,306,0.8361
3,0,Negative,Facility,503,182,133,830,0.1146
4,1,Neutral,Facility,155,121,56,140,0.0317
5,2,Positive,Facility,6801,9585,936,658,1.1190
6,0,Negative,Service,539,166,106,799,0.1072
7,1,Neutral,Service,152,121,47,99,0.0267
8,2,Positive,Service,8437,16988,632,331,1.7969


## 3. Data Cleaning và EDA


### 3.1 Kiểm định tính toàn vẹn dữ liệu và kiểm tra Missing Values


In [16]:
missing_reviews = df_reviews.isnull().sum()
missing_ann = df_annotations.isnull().sum()
print("Missing values trong df_reviews:\n", missing_reviews[missing_reviews > 0])
print("Missing values trong df_annotations:\n", missing_ann[missing_ann > 0])

invalid_spans = df_annotations[
    (df_annotations['start_char'] < 0) |
    (df_annotations['start_char'] >= df_annotations['end_char'])
]
print(f"Số lượng annotation spans có tọa độ không hợp lệ: {len(invalid_spans)}")

sns.set_theme(style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

fields = ["hotel_name", "location", "star", "overall_sentiment", "date", "review_text"]
field_labels = ["Hotel Name", "Location", "Star Rating", "Overall Sentiment", "Review Date", "Review Text"]
completeness = (1 - df_reviews[fields].isnull().mean()) * 100

sns.barplot(x=completeness.values, y=field_labels, ax=axes[0], color="#2b5c8f", edgecolor="#1a365d", linewidth=0.8)
axes[0].set_xlabel("Completeness Rate (%)", fontsize=11, labelpad=8)
axes[0].set_ylabel("Metadata Field", fontsize=11, labelpad=8)
axes[0].set_xlim(0, 118)
axes[0].tick_params(axis="both", labelsize=10)
axes[0].grid(axis="x", linestyle="--", alpha=0.3)
sns.despine(ax=axes[0])
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_width():.1f}%", (p.get_width() + 1.5, p.get_y() + p.get_height() / 2),
                     va="center", fontsize=9.5, fontweight="normal")

sns.histplot(df_reviews["word_count"], bins=50, kde=True, ax=axes[1], color="#d35400", edgecolor="#ba4a00", linewidth=0.5)
median_wc = df_reviews["word_count"].median()
mean_wc = df_reviews["word_count"].mean()
axes[1].axvline(median_wc, color="#c0392b", linestyle="--", linewidth=1.8, label=f"Median: {median_wc:.0f} words")
axes[1].axvline(mean_wc, color="#2c3e50", linestyle=":", linewidth=1.8, label=f"Mean: {mean_wc:.0f} words")
axes[1].set_xlabel("Word Count per Review", fontsize=11, labelpad=8)
axes[1].set_ylabel("Frequency", fontsize=11, labelpad=8)
axes[1].tick_params(axis="both", labelsize=10)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)
axes[1].legend(frameon=True, facecolor="white", framealpha=0.9, fontsize=9.5, loc="upper right")
sns.despine(ax=axes[1])

plt.tight_layout()
chart_paths_31 = [
    "docs/main_paper/chart_img/31_missing_values_and_data_summary.png"
]
for p in chart_paths_31:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ làm sạch dữ liệu tại: {chart_paths_31}")

Missing values trong df_reviews:
 Series([], dtype: int64)
Missing values trong df_annotations:
 Series([], dtype: int64)
Số lượng annotation spans có tọa độ không hợp lệ: 0
Đã lưu biểu đồ làm sạch dữ liệu tại: 'data/chart_img/31_missing_values_and_data_summary.png'


### 3.2 Thống kê mô tả đặc tính văn bản và phân vị độ dài


In [17]:
desc_stats = df_reviews[['char_length', 'word_count', 'sentiment_id']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
print("Bảng thống kê mô tả đặc tính văn bản đánh giá:")
display(desc_stats)

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(df_reviews["char_length"], fill=True, color="#27ae60", alpha=0.35, linewidth=2, ax=ax)

median_cl = df_reviews["char_length"].median()
mean_cl = df_reviews["char_length"].mean()
p95_cl = df_reviews["char_length"].quantile(0.95)

ax.axvline(median_cl, color="#c0392b", linestyle="--", linewidth=1.8, label=f"Median: {median_cl:.0f} chars")
ax.axvline(mean_cl, color="#2980b9", linestyle=":", linewidth=1.8, label=f"Mean: {mean_cl:.0f} chars")
ax.axvline(p95_cl, color="#8e44ad", linestyle="-.", linewidth=1.5, label=f"95th Pct: {p95_cl:.0f} chars")

ax.set_xlabel("Review Character Length (Characters)", fontsize=11, labelpad=8)
ax.set_ylabel("Kernel Density Estimate (KDE)", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="both", linestyle="--", alpha=0.3)
ax.legend(frameon=True, facecolor="white", framealpha=0.9, fontsize=9.5, loc="upper right")
sns.despine(ax=ax)

plt.tight_layout()
chart_paths_32 = [
    "docs/main_paper/chart_img/32_review_text_length_distribution.png"
]
for p in chart_paths_32:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ phân bố độ dài tại: {chart_paths_32}")

Bảng thống kê mô tả đặc tính văn bản đánh giá:


,count,mean,std,min,5%,25%,50%,75%,95%,max
char_length,9990.0,450.664464,200.472604,177.0,210.45,274.0,398.0,618.0,786.0,847.0
word_count,9990.0,82.145846,37.112671,30.0,37.00,49.0,72.0,114.0,146.0,150.0
sentiment_id,9990.0,1.834835,0.521197,0.0,0.00,2.0,2.0,2.0,2.0,2.0


Đã lưu biểu đồ phân bố độ dài tại: 'data/chart_img/32_review_text_length_distribution.png'


## 4. Data Visualization


### 4.1 Phân bố cảm xúc tổng quan 3 lớp (Negative, Neutral, Positive)


In [4]:
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8, 5))
sent_order = ["Negative", "Neutral", "Positive"]
sent_counts = df_reviews["overall_sentiment"].value_counts().reindex(sent_order, fill_value=0)
colors_41 = ["#c0392b", "#e67e22", "#27ae60"]
bars = ax.bar(sent_counts.index, sent_counts.values, color=colors_41, width=0.52, edgecolor="#2c3e50", linewidth=0.8)

ax.set_xlabel("Sentiment Class", fontsize=11, labelpad=8)
ax.set_ylabel("Number of Reviews", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df_reviews)) * 100
    ax.text(bar.get_x() + bar.get_width() / 2.0, yval + (max(sent_counts.values) * 0.015),
            f"{yval:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10, fontweight="normal")

ax.set_ylim(0, max(sent_counts.values) * 1.15)
sns.despine(ax=ax)
plt.tight_layout()

chart_paths_41 = [
    "docs/main_paper/chart_img/41_overall_rating_distribution.png"
]
for p in chart_paths_41:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_41}")

Đã lưu biểu đồ: 'data/chart_img/41_overall_rating_distribution.png'


### 4.2 Top 10 khách sạn có số review cao nhất


In [5]:
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(11, 5.5))
top_hotels = df_reviews["hotel_name"].value_counts().head(10)
hotel_labels = [unicodedata.normalize("NFC", name).strip().title() for name in top_hotels.index]
hotel_pos_ratio = df_reviews.groupby("hotel_name")["sentiment_id"].apply(lambda s: (s == 2).mean() * 100)

sns.barplot(x=top_hotels.values, y=hotel_labels, palette="Blues_r", hue=hotel_labels, legend=False,
            ax=ax, edgecolor="#1a365d", linewidth=0.6)

ax.set_xlabel("Number of Reviews", fontsize=11, labelpad=8)
ax.set_ylabel("Hotel Name", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=9.5)
ax.grid(axis="x", linestyle="--", alpha=0.3)

for i, (orig_name, v) in enumerate(top_hotels.items()):
    pos_r = hotel_pos_ratio.get(orig_name, 0.0)
    pct = (v / len(df_reviews)) * 100
    ax.text(v + (max(top_hotels.values) * 0.01), i, f"{v:,} ({pct:.1f}%) | Pos: {pos_r:.1f}%",
            va="center", fontsize=9.5, fontweight="normal")

ax.set_xlim(0, max(top_hotels.values) * 1.35)
sns.despine(ax=ax)
plt.tight_layout()

chart_paths_42 = [
    "docs/main_paper/chart_img/42_top10_hotels_highest_reviews.png"
]
for p in chart_paths_42:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_42}")

Đã lưu biểu đồ: 'data/chart_img/42_top10_hotels_highest_reviews.png'


### 4.3 Tần suất xuất hiện của các khía cạnh khách sạn


In [6]:
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(9, 5))
aspect_counts = df_annotations["aspect"].value_counts()

sns.barplot(x=aspect_counts.index, y=aspect_counts.values, palette="viridis", hue=aspect_counts.index, legend=False,
            ax=ax, edgecolor="#2c3e50", linewidth=0.7)

ax.set_xlabel("Hotel Aspect Category", fontsize=11, labelpad=8)
ax.set_ylabel("Annotation Frequency", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)

total_ann = aspect_counts.sum()
for i, v in enumerate(aspect_counts.values):
    pct = (v / total_ann) * 100
    ax.text(i, v + (max(aspect_counts.values) * 0.015), f"{v:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=9.5, fontweight="normal")

ax.set_ylim(0, max(aspect_counts.values) * 1.15)
sns.despine(ax=ax)
plt.tight_layout()

chart_paths_43 = [
    "docs/main_paper/chart_img/43_aspect_frequency_distribution.png"
]
for p in chart_paths_43:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_43}")

Đã lưu biểu đồ: 'data/chart_img/43_aspect_frequency_distribution.png'


### 4.4 Phân bố cảm xúc theo từng khía cạnh


In [7]:
sns.set_theme(style="ticks")
asp_sent = pd.crosstab(df_annotations["aspect"], df_annotations["sentiment"], normalize="index") * 100
cols = [c for c in ["Positive", "Neutral", "Negative"] if c in asp_sent.columns]
asp_sent = asp_sent[cols]
colors_44 = {"Positive": "#2e7d32", "Neutral": "#f9a825", "Negative": "#c62828"}

fig, ax = plt.subplots(figsize=(10, 5))
asp_sent.plot(kind="bar", stacked=True, color=[colors_44[c] for c in cols], edgecolor="#ffffff", linewidth=0.8, ax=ax)

ax.set_xlabel("Hotel Aspect Category", fontsize=11, labelpad=8)
ax.set_ylabel("Proportion (%)", fontsize=11, labelpad=8)
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0, labelsize=10)
ax.tick_params(axis="y", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)

for p in ax.patches:
    h = p.get_height()
    if h >= 6.0:
        ax.text(p.get_x() + p.get_width() / 2.0, p.get_y() + h / 2.0,
                f"{h:.1f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")

ax.legend(title="Sentiment", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True, facecolor="white", framealpha=0.9,
          fontsize=9.5, title_fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()

chart_paths_44 = [
    "docs/main_paper/chart_img/44_aspect_sentiment_breakdown.png"
]
for p in chart_paths_44:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_44}")

Đã lưu biểu đồ: 'data/chart_img/44_aspect_sentiment_breakdown.png'


### 4.5 Độ dài đoạn bằng chứng theo khía cạnh


In [8]:
df_annotations["span_len"] = df_annotations["end_char"] - df_annotations["start_char"]
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(data=df_annotations, x="aspect", y="span_len", palette="Set2", hue="aspect", legend=False,
            showfliers=False, ax=ax, linewidth=1.2, width=0.55)

ax.set_xlabel("Hotel Aspect Category", fontsize=11, labelpad=8)
ax.set_ylabel("Rationale Span Length (Characters)", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=ax)

plt.tight_layout()
chart_paths_45 = [
    "docs/main_paper/chart_img/45_rationale_length_by_aspect.png"
]
for p in chart_paths_45:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_45}")

Đã lưu biểu đồ: 'data/chart_img/45_rationale_length_by_aspect.png'


### 4.6 Ma trận tương quan đồng xuất hiện giữa các khía cạnh


In [9]:
aspect_list = sorted(df_annotations["aspect"].unique())
co_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)

rev_aspects = df_annotations.groupby("review_id")["aspect"].unique()
for asps in rev_aspects:
    for a1 in asps:
        for a2 in asps:
            co_mat.loc[a1, a2] += 1

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(co_mat, annot=True, fmt="d", cmap="YlGnBu", cbar=True, ax=ax,
            annot_kws={"fontsize": 10, "fontweight": "normal"},
            cbar_kws={"label": "Co-occurrence Frequency", "shrink": 0.85},
            linewidths=0.5, linecolor="#eceff1")

ax.set_xlabel("Hotel Aspect", fontsize=11, labelpad=8)
ax.set_ylabel("Hotel Aspect", fontsize=11, labelpad=8)
ax.tick_params(axis="x", rotation=25, labelsize=10)
ax.tick_params(axis="y", rotation=0, labelsize=10)

plt.tight_layout()
chart_paths_46 = [
    "docs/main_paper/chart_img/46_aspect_cooccurrence_matrix.png"
]
for p in chart_paths_46:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_46}")

Đã lưu biểu đồ: 'data/chart_img/46_aspect_cooccurrence_matrix.png'


### 4.7 Tỷ lệ phân kỳ cảm xúc giữa các cặp khía cạnh trong cùng review


In [10]:
conflict_mat = pd.DataFrame(0, index=aspect_list, columns=aspect_list)
rev_asp_sent = df_annotations.groupby(["review_id", "aspect"])["sentiment"].unique()

rev_to_asp = defaultdict(dict)
for (rid, asp), sents in rev_asp_sent.items():
    rev_to_asp[rid][asp] = sents

for rid, asps_dict in rev_to_asp.items():
    for a1, s1 in asps_dict.items():
        for a2, s2 in asps_dict.items():
            if a1 != a2 and "Positive" in s1 and "Negative" in s2:
                conflict_mat.loc[a1, a2] += 1

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(conflict_mat, annot=True, fmt="d", cmap="Reds", cbar=True, ax=ax,
            annot_kws={"fontsize": 10, "fontweight": "normal"},
            cbar_kws={"label": "Conflicting Review Count", "shrink": 0.85},
            linewidths=0.5, linecolor="#eceff1")

ax.set_xlabel("Negative Sentiment Aspect", fontsize=11, labelpad=8)
ax.set_ylabel("Positive Sentiment Aspect", fontsize=11, labelpad=8)
ax.tick_params(axis="x", rotation=25, labelsize=10)
ax.tick_params(axis="y", rotation=0, labelsize=10)

plt.tight_layout()
chart_paths_47 = [
    "docs/main_paper/chart_img/47_sentiment_divergence_rate.png"
]
for p in chart_paths_47:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_47}")

Đã lưu biểu đồ: 'data/chart_img/47_sentiment_divergence_rate.png'


### 4.8 Phân bố số lượng từ của bài đánh giá theo nhãn cảm xúc 3 lớp


In [11]:
sent_order = ["Negative", "Neutral", "Positive"]
df_plot_48 = df_reviews.dropna(subset=["overall_sentiment", "word_count"]).copy()
df_plot_48 = df_plot_48[df_plot_48["overall_sentiment"].isin(sent_order)]

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8.5, 5.2))
colors_48 = ["#c0392b", "#e67e22", "#27ae60"]

sns.boxplot(
    data=df_plot_48, x="overall_sentiment", y="word_count",
    order=sent_order, hue="overall_sentiment", hue_order=sent_order,
    palette=colors_48, legend=False, showfliers=False, ax=ax,
    linewidth=1.2, width=0.5
)

# Calculate and annotate median word count for each sentiment class
medians = df_plot_48.groupby("overall_sentiment")["word_count"].median().reindex(sent_order)
for i, (sent, med) in enumerate(medians.items()):
    ax.text(i, med + 4, f"Median: {med:.0f}", ha="center", va="bottom", fontsize=9.5,
            fontweight="bold", color="#1a252f")

ax.set_xlabel("Overall Sentiment Class", fontsize=11, labelpad=8)
ax.set_ylabel("Word Count per Review", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)
sns.despine(ax=ax)

plt.tight_layout()
chart_paths_48 = [
    "docs/main_paper/chart_img/48_review_wordcount_by_star.png"
]
for p in chart_paths_48:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_48}")

Đã lưu biểu đồ: 'data/chart_img/48_review_wordcount_by_star.png'


### 4.9 Mức độ chồng lấn tọa độ văn bản giữa các khía cạnh


In [12]:
collision_records = []
ann_grouped = df_annotations.groupby("review_id")

for rid, group in ann_grouped:
    records = group[["annotation_id", "aspect", "start_char", "end_char"]].to_dict("records")
    n = len(records)
    for i in range(n):
        for j in range(i + 1, n):
            r1, r2 = records[i], records[j]
            if r1["aspect"] != r2["aspect"]:
                if not (r1["end_char"] <= r2["start_char"] or r1["start_char"] >= r2["end_char"]):
                    pair_label = f"{min(r1['aspect'], r2['aspect'])} & {max(r1['aspect'], r2['aspect'])}"
                    collision_records.append(pair_label)

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(8.5, 4.8))
if collision_records:
    col_counts = pd.Series(collision_records).value_counts()
    sns.barplot(x=col_counts.values, y=col_counts.index, color="#c0392b", ax=ax,
                edgecolor="#7b241c", linewidth=0.7)
    ax.set_xlabel("Number of Overlapping Text Coordinate Spans", fontsize=11, labelpad=8)
    ax.set_ylabel("Aspect Pair", fontsize=11, labelpad=8)
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    for i, v in enumerate(col_counts.values):
        ax.text(v + 0.08, i, str(v), va="center", fontsize=10, fontweight="bold")
    ax.set_xlim(0, max(col_counts.values) + 0.6)
else:
    ax.text(0.5, 0.5, "No text coordinate collisions detected", ha="center", va="center", fontsize=11)
sns.despine(ax=ax)

plt.tight_layout()
chart_paths_49 = [
    "docs/main_paper/chart_img/49_cross_aspect_span_overlap.png"
]
for p in chart_paths_49:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_49}")

Đã lưu biểu đồ: 'data/chart_img/49_cross_aspect_span_overlap.png'


### 4.10 Phân bố số lượng khía cạnh được đề cập trên mỗi bài đánh giá


In [13]:
aspects_per_rev = df_annotations.groupby("review_id")["aspect"].nunique()
full_aspects_per_rev = df_reviews["review_id"].map(aspects_per_rev).fillna(0).astype(int)

sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(9, 5))
counts_series = full_aspects_per_rev.value_counts().sort_index()

sns.barplot(x=counts_series.index, y=counts_series.values, palette="mako", hue=counts_series.index, legend=False,
            ax=ax, edgecolor="#1a365d", linewidth=0.7)

ax.set_xlabel("Number of Distinct Aspects", fontsize=11, labelpad=8)
ax.set_ylabel("Number of Reviews", fontsize=11, labelpad=8)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)

for i, v in enumerate(counts_series.values):
    pct = (v / len(df_reviews)) * 100
    ax.text(i, v + (max(counts_series.values) * 0.015), f"{v:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=9.5, fontweight="normal")

ax.set_ylim(0, max(counts_series.values) * 1.15)
sns.despine(ax=ax)

plt.tight_layout()
chart_paths_410 = [
    "docs/main_paper/chart_img/410_aspects_per_review_distribution.png"
]
for p in chart_paths_410:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    plt.savefig(p, dpi=300, bbox_inches="tight", transparent=False, facecolor="white")
plt.close()
print(f"Đã lưu biểu đồ: {chart_paths_410}")

Đã lưu biểu đồ: 'data/chart_img/410_aspects_per_review_distribution.png'


### 4.11 Sơ đồ Quy trình Nghiên cứu (Research Pipeline)


In [1]:
import os
import shutil
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("docs/main_paper/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.22, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.04, y - 0.045), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#000000", alpha=0.055, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.40 * scale
    h = 0.20 * scale
    for idx, dy in enumerate([-0.16 * scale, 0.0, 0.16 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.05 * scale],
                [cx + w + 0.05 * scale, y0],
                [cx, y0 - h - 0.05 * scale],
                [cx - w - 0.05 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.8, zorder=zorder + idx*2 + 1))

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.48 * scale, 0.62 * scale
    x0, y0 = cx - dw / 2 - 0.03 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.08*scale}",
        facecolor="none", edgecolor=color, linewidth=3.0, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.10*scale], [y0 + dh - 0.17*scale, y0 + dh - 0.17*scale],
            color=color, lw=2.8, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.10*scale, x0 + dw - 0.17*scale], [y0 + dh - 0.31*scale, y0 + dh - 0.31*scale],
            color=color, lw=2.8, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.05*scale, y0 + 0.11*scale
    r = 0.135 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.04*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=3.0, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.55], [mcy - r*0.7, mcy - r*1.55],
            color=color, lw=3.2, solid_capstyle="round", zorder=zorder+3)

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.30 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=3.2, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.09*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.08*scale, ay_pt + 0.10*scale],
        [ax_pt - 0.05*scale, ay_pt - 0.09*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=1.2, zorder=zorder+1))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.56 * scale, 0.48 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=3.0, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.13*scale, y0 + 0.17*scale),
        (x0 + 0.30*scale, y0 + 0.33*scale),
        (x0 + 0.49*scale, y0 + 0.23*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.8, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.052*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.8, zorder=zorder+2))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.58 * scale, 0.12 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.06*scale}",
        facecolor="none", edgecolor=color, linewidth=2.8, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.34*scale, cy + 0.12*scale], color=color, lw=2.8, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.10*scale, cx, cx + 0.10*scale], [cy + 0.22*scale, cy + 0.11*scale, cy + 0.22*scale],
            color=color, lw=2.8, solid_capstyle="round", solid_joinstyle="round", zorder=zorder+1)
    ax.add_patch(Circle((cx + 0.10*scale, cy), 0.062*scale, facecolor=color, edgecolor="#FFFFFF", lw=1.5, zorder=zorder+2))

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    color = "#0099C6"
    layer_xs = [cx - 0.32 * scale, cx, cx + 0.32 * scale]
    node_ys = [
        [cy - 0.26 * scale, cy, cy + 0.26 * scale],
        [cy - 0.36 * scale, cy - 0.12 * scale, cy + 0.12 * scale, cy + 0.36 * scale],
        [cy - 0.26 * scale, cy, cy + 0.26 * scale]
    ]
    for l_idx in range(len(layer_xs) - 1):
        for y_a in node_ys[l_idx]:
            for y_b in node_ys[l_idx + 1]:
                ax.plot([layer_xs[l_idx], layer_xs[l_idx + 1]], [y_a, y_b],
                        color=color, lw=1.2, alpha=0.55, zorder=zorder)
    for l_idx, lx in enumerate(layer_xs):
        for ny in node_ys[l_idx]:
            ax.add_patch(Circle((lx, ny), 0.058 * scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + 1))
            ax.add_patch(Circle((lx, ny), 0.048 * scale, facecolor=color, edgecolor="#FFFFFF", linewidth=1.4, zorder=zorder + 2))

def build_figure_411():
    W, H = 19.2, 4.60
    fig, ax = plt.subplots(figsize=(W, H), dpi=300)
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.axis("off")

    start_x = 0.55
    card_w = 2.50
    gap = 0.26
    card_y = 0.90
    card_h = 2.80

    steps = [
        ("Data Extraction", "Database & DF", draw_icon_data_layers),
        ("Preprocessing", "NLTK & Cleaning", draw_icon_preprocessing_doc),
        ("EDA & RQs", "SQL & Stats", draw_icon_circular_arrow),
        ("Multi-Task", "3-Class Polarity", draw_icon_chart_nodes),
        ("Optimization", "Overlap Penalty", draw_icon_optimization),
    ]

    for i, (title, subtitle, icon_fn) in enumerate(steps):
        cx_box = start_x + i * (card_w + gap)
        draw_rounded_card(ax, cx_box, card_y, card_w, card_h,
                          bg="#FFFFFF", edgecolor="#DCDCE2", lw=1.2, radius=0.22, shadow=True, zorder=3)
        center_x = cx_box + card_w / 2
        
        # Icon positioned with ample headroom
        icon_fn(ax, center_x, card_y + 1.84, scale=1.12, zorder=6)
        
        # Title and Subtitle with clear vertical separation and generous padding from edges
        ax.text(center_x, card_y + 1.05, title, ha="center", va="center",
                fontsize=13.5, fontweight="bold", color="#1D1D1F", zorder=7)
        ax.text(center_x, card_y + 0.60, subtitle, ha="center", va="center",
                fontsize=11.2, fontweight="bold", color="#515154", zorder=7)

        # Arrow between cards
        if i < len(steps) - 1:
            x1 = cx_box + card_w
            x2 = cx_box + card_w + gap
            ym = card_y + card_h / 2
            ax.plot([x1, x2], [ym, ym], color="#86868B", lw=2.4, zorder=2)
            ax.plot([x2 - 0.07, x2, x2 - 0.07], [ym + 0.07, ym, ym - 0.07],
                    color="#86868B", lw=2.4, solid_capstyle="round", zorder=2)

    last_card_right = start_x + 4 * (card_w + gap) + card_w
    right_box_x = 14.36
    right_box_y = 0.20
    right_box_w = 4.28
    right_box_h = 4.20
    ym_arrow = card_y + card_h / 2
    
    # Connecting arrow to right container
    ax.plot([last_card_right, right_box_x - 0.04], [ym_arrow, ym_arrow], color="#86868B", lw=2.4, zorder=2)
    ax.plot([right_box_x - 0.16, right_box_x - 0.03, right_box_x - 0.16],
            [ym_arrow + 0.10, ym_arrow, ym_arrow - 0.10], color="#86868B", lw=2.4, solid_capstyle="round", zorder=2)

    draw_rounded_card(ax, right_box_x, right_box_y, right_box_w, right_box_h,
                      bg="#F8FAFC", edgecolor="#FFFFFF", lw=2.2, radius=0.26, shadow=True, zorder=3)
    rb_border = FancyBboxPatch(
        (right_box_x, right_box_y), right_box_w, right_box_h,
        boxstyle="round,pad=0.0,rounding_size=0.26",
        facecolor="none", edgecolor="#D8D8DE", linewidth=1.3, zorder=4
    )
    ax.add_patch(rb_border)

    rcx = right_box_x + right_box_w / 2
    ax.text(rcx, right_box_y + right_box_h - 0.36, "6-Aspect Heads", ha="center", va="center",
            fontsize=16.5, fontweight="bold", color="#1D1D1F", zorder=7)

    aspect_names = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    bar_w = 0.51
    bar_h = 1.35
    bar_gap = 0.11
    total_bars_w = 6 * bar_w + 5 * bar_gap
    bar_start_x = rcx - total_bars_w / 2
    bar_y = right_box_y + 2.18
    bar_centers = []

    for a_i, a_lbl in enumerate(aspect_names):
        bx = bar_start_x + a_i * (bar_w + bar_gap)
        bcx = bx + bar_w / 2
        bar_centers.append(bcx)
        pill = FancyBboxPatch(
            (bx, bar_y), bar_w, bar_h,
            boxstyle="round,pad=0.0,rounding_size=0.09",
            facecolor="#0099C6", edgecolor="none", zorder=5
        )
        ax.add_patch(pill)
        for d_i in range(3):
            ax.add_patch(Circle((bcx, bar_y + bar_h - 0.12 - d_i * 0.065), 0.020, facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, bar_y + 0.50, a_lbl, ha="center", va="center", rotation=90,
                fontsize=11.2, fontweight="bold", color="#FFFFFF", zorder=7)

    y_tree_top = bar_y
    y_tree_horiz = bar_y - 0.16
    y_tree_bot = bar_y - 0.38
    for bcx in bar_centers:
        ax.plot([bcx, bcx], [y_tree_top, y_tree_horiz], color="#0099C6", lw=2.2, zorder=5)
    ax.plot([bar_centers[0], bar_centers[-1]], [y_tree_horiz, y_tree_horiz], color="#0099C6", lw=2.2, zorder=5)
    ax.plot([rcx, rcx], [y_tree_horiz, y_tree_bot], color="#0099C6", lw=2.2, zorder=5)

    draw_icon_neural_network(ax, rcx, right_box_y + 1.34, scale=1.05, zorder=6)

    ax.text(rcx, right_box_y + 0.68, "CRAM-ABSA Model", ha="center", va="center",
            fontsize=16.5, fontweight="bold", color="#1D1D1F", zorder=7)
    ax.text(rcx, right_box_y + 0.36, "3-Class: Pos / Neu / Neg", ha="center", va="center",
            fontsize=11.6, fontweight="bold", color="#515154", zorder=7)

    out_p = "docs/main_paper/chart_img/411_research_pipeline.png"
    plt.savefig(out_p, dpi=300, bbox_inches="tight", pad_inches=0.04, facecolor="white")
    plt.close(fig)
    print(f"Generated {out_p} successfully.")

build_figure_411()


Generated data/chart_img/411_research_pipeline.png and synced to docs/main_paper/chart_img/411_research_pipeline.png successfully.


### 4.12 Kiến trúc Tổng thể Đề xuất (Abstract Architecture of Proposed Method)


In [15]:
import os
import shutil
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Polygon, Arc

os.makedirs("docs/main_paper/chart_img", exist_ok=True)

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
plt.rcParams["font.family"] = "sans-serif"

def draw_rounded_card(ax, x, y, w, h, bg="#FFFFFF", edgecolor="none", lw=1.0, radius=0.20, shadow=True, zorder=3):
    if shadow:
        sh = FancyBboxPatch(
            (x + 0.035, y - 0.05), w, h,
            boxstyle=f"round,pad=0.0,rounding_size={radius}",
            facecolor="#0F172A", alpha=0.06, edgecolor="none", zorder=zorder - 1
        )
        ax.add_patch(sh)
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.0,rounding_size={radius}",
        facecolor=bg, edgecolor=edgecolor, linewidth=lw, zorder=zorder
    )
    ax.add_patch(box)

def draw_step_arrow(ax, x1, x2, y, color="#64748B", lw=2.2, head_w=0.10, head_h=0.07, zorder=4):
    ax.plot([x1, x2 - 0.02], [y, y], color=color, lw=lw, solid_capstyle="round", zorder=zorder)
    ax.plot([x2 - head_w, x2 - 0.01, x2 - head_w],
            [y + head_h, y, y - head_h], color=color, lw=lw, solid_capstyle="round", zorder=zorder)

def draw_down_arrow(ax, x, y1, y2, color="#64748B", lw=2.2, head_w=0.08, head_h=0.10, zorder=4):
    ax.plot([x, x], [y1, y2 + 0.02], color=color, lw=lw, solid_capstyle="round", zorder=zorder)
    ax.plot([x - head_w, x, x + head_w],
            [y2 + head_h, y2 + 0.01, y2 + head_h], color=color, lw=lw, solid_capstyle="round", zorder=zorder)

def draw_icon_preprocessing_doc(ax, cx, cy, scale=1.0, zorder=6):
    color = "#8E6CEF"
    dw, dh = 0.40 * scale, 0.48 * scale
    x0, y0 = cx - dw / 2 - 0.02 * scale, cy - dh / 2
    doc = FancyBboxPatch(
        (x0, y0), dw, dh,
        boxstyle=f"round,pad=0.0,rounding_size={0.06*scale}",
        facecolor="none", edgecolor=color, linewidth=2.4, zorder=zorder
    )
    ax.add_patch(doc)
    ax.plot([x0 + 0.09*scale, x0 + dw - 0.09*scale], [y0 + dh - 0.14*scale, y0 + dh - 0.14*scale],
            color=color, lw=2.2, solid_capstyle="round", zorder=zorder+1)
    ax.plot([x0 + 0.09*scale, x0 + dw - 0.14*scale], [y0 + dh - 0.24*scale, y0 + dh - 0.24*scale],
            color=color, lw=2.2, solid_capstyle="round", zorder=zorder+1)
    mcx, mcy = x0 + dw - 0.04*scale, y0 + 0.10*scale
    r = 0.11 * scale
    ax.add_patch(Circle((mcx, mcy), r + 0.03*scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder+2))
    ax.add_patch(Circle((mcx, mcy), r, facecolor="#FFFFFF", edgecolor=color, linewidth=2.4, zorder=zorder+3))
    ax.plot([mcx + r*0.7, mcx + r*1.5], [mcy - r*0.7, mcy - r*1.5],
            color=color, lw=2.6, solid_capstyle="round", zorder=zorder+3)

def draw_icon_data_layers(ax, cx, cy, scale=1.0, zorder=6):
    color = "#2997FF"
    w = 0.30 * scale
    h = 0.14 * scale
    for idx, dy in enumerate([-0.12 * scale, 0.0, 0.12 * scale]):
        y0 = cy + dy
        pts = np.array([
            [cx, y0 + h],
            [cx + w, y0],
            [cx, y0 - h],
            [cx - w, y0]
        ])
        if idx < 2:
            pts_halo = np.array([
                [cx, y0 + h + 0.04 * scale],
                [cx + w + 0.04 * scale, y0],
                [cx, y0 - h - 0.04 * scale],
                [cx - w - 0.04 * scale, y0]
            ])
            ax.add_patch(Polygon(pts_halo, closed=True, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + idx*2))
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, edgecolor="#FFFFFF", linewidth=1.4, zorder=zorder + idx*2 + 1))

def draw_icon_neural_network(ax, cx, cy, scale=1.0, zorder=6):
    color = "#1DB096"
    layer_xs = [cx - 0.20 * scale, cx, cx + 0.20 * scale]
    node_ys = [
        [cy - 0.18 * scale, cy, cy + 0.18 * scale],
        [cy - 0.24 * scale, cy - 0.08 * scale, cy + 0.08 * scale, cy + 0.24 * scale],
        [cy - 0.18 * scale, cy, cy + 0.18 * scale]
    ]
    for l_idx in range(len(layer_xs) - 1):
        for y_a in node_ys[l_idx]:
            for y_b in node_ys[l_idx + 1]:
                ax.plot([layer_xs[l_idx], layer_xs[l_idx + 1]], [y_a, y_b],
                        color=color, lw=0.9, alpha=0.55, zorder=zorder)
    for l_idx, lx in enumerate(layer_xs):
        for ny in node_ys[l_idx]:
            ax.add_patch(Circle((lx, ny), 0.044 * scale, facecolor="#FFFFFF", edgecolor="none", zorder=zorder + 1))
            ax.add_patch(Circle((lx, ny), 0.038 * scale, facecolor=color, edgecolor="#FFFFFF", linewidth=1.1, zorder=zorder + 2))

def draw_icon_circular_arrow(ax, cx, cy, scale=1.0, zorder=6):
    color = "#F5428D"
    r = 0.22 * scale
    arc = Arc((cx, cy), 2*r, 2*r, angle=0, theta1=75, theta2=395, color=color, lw=2.6, capstyle="round", zorder=zorder)
    ax.add_patch(arc)
    ang = math.radians(68)
    ax_pt = cx + r * math.cos(ang)
    ay_pt = cy + r * math.sin(ang)
    head = np.array([
        [ax_pt + 0.07*scale, ay_pt + 0.01*scale],
        [ax_pt - 0.06*scale, ay_pt + 0.08*scale],
        [ax_pt - 0.03*scale, ay_pt - 0.07*scale]
    ])
    ax.add_patch(Polygon(head, closed=True, facecolor=color, edgecolor=color, lw=0.9, zorder=zorder+1))

def draw_icon_optimization(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF9500"
    bw, bh = 0.44 * scale, 0.10 * scale
    bar = FancyBboxPatch(
        (cx - bw/2, cy - bh/2), bw, bh,
        boxstyle=f"round,pad=0.0,rounding_size={0.045*scale}",
        facecolor="none", edgecolor=color, linewidth=2.3, zorder=zorder
    )
    ax.add_patch(bar)
    ax.plot([cx, cx], [cy + 0.26*scale, cy + 0.10*scale], color=color, lw=2.3, solid_capstyle="round", zorder=zorder+1)
    ax.plot([cx - 0.08*scale, cx, cx + 0.08*scale], [cy + 0.18*scale, cy + 0.09*scale, cy + 0.18*scale],
            color=color, lw=2.3, solid_capstyle="round", solid_joinstyle="round", zorder=zorder+1)
    ax.add_patch(Circle((cx + 0.08*scale, cy), 0.05*scale, facecolor=color, edgecolor="#FFFFFF", lw=1.2, zorder=zorder+2))

def draw_icon_chart_nodes(ax, cx, cy, scale=1.0, zorder=6):
    color = "#FF5733"
    w, h = 0.42 * scale, 0.36 * scale
    x0, y0 = cx - w/2, cy - h/2
    ax.plot([x0, x0, x0 + w], [y0 + h, y0, y0], color=color, lw=2.4, solid_capstyle="round", zorder=zorder)
    pts = [
        (x0 + 0.10*scale, y0 + 0.13*scale),
        (x0 + 0.23*scale, y0 + 0.25*scale),
        (x0 + 0.37*scale, y0 + 0.17*scale)
    ]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=color, lw=2.2, zorder=zorder+1)
    for px, py in pts:
        ax.add_patch(Circle((px, py), 0.040*scale, facecolor="#FFFFFF", edgecolor=color, lw=2.0, zorder=zorder+2))

def build_figure_412():
    W, H = 19.4, 9.8
    fig, ax = plt.subplots(figsize=(W, H), dpi=300)
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.axis("off")

    # Outer container
    draw_rounded_card(ax, 0.15, 0.15, 19.10, 9.50, bg="#F8FAFC", edgecolor="#DCE0E8", lw=1.5, radius=0.28, shadow=False, zorder=1)

    stages_row1 = [
        {
            "num": "Stage 1",
            "title": "Dual-Stream Input",
            "subtitle": "Text Cleaning & Negation NLTK",
            "color": "#8E6CEF", "tint": "#F4F0FE", "border": "#DDD3FB",
            "icon": draw_icon_preprocessing_doc,
            "rows": [
                ("NFC & Contraction Expansion", "can't -> cannot, won't -> will not"),
                ("Negation-Safe Stopwords", "SAFE_STOPWORDS = NLTK \\ {not, never, but}"),
                ("Dual Text Streams", "clean_review_text (BERT) & paper1_nltk_text"),
            ],
            "footer": "Out: Dual Streams (Syntax + NLTK Tokens)",
        },
        {
            "num": "Stage 2",
            "title": "20-Feature & EBR",
            "subtitle": "Parquet Store | 20 Engineered Feats",
            "color": "#2997FF", "tint": "#EEF6FF", "border": "#C9E2FF",
            "icon": draw_icon_data_layers,
            "rows": [
                ("Structural & Span (F01–F07)", "aspect_count, density, cross_overlap"),
                ("Polarity & Hedging (F08–F19)", "6-aspect s_a, conflict, hedging, negation"),
                ("EBR Prototype Sim (F20)", "Top-3 dense cosine sim with ASPECT_KEYWORDS"),
            ],
            "footer": "Out: Feature Vector f in R^20 (Parquet)",
        },
        {
            "num": "Stage 3",
            "title": "BERT Backbone",
            "subtitle": "bert-base-uncased (768d)",
            "color": "#1DB096", "tint": "#ECFAF6", "border": "#BDEBDF",
            "icon": draw_icon_neural_network,
            "rows": [
                ("WordPiece Tokenization", "[CLS] + clean_review_text + [SEP] (L <= 256)"),
                ("12-Layer Transformer", "12 Self-Attention Heads, d_h = 768"),
                ("Contextual Hidden States", "Token matrix H in R^(L x 768) & global h_[CLS]"),
            ],
            "footer": "Out: H in R^(L x 768), h_[CLS] in R^768",
        },
    ]

    stages_row2 = [
        {
            "num": "Stage 4",
            "title": "Feature Fusion",
            "subtitle": "Residual Gate (20d -> 768d)",
            "color": "#F5428D", "tint": "#FEF0F6", "border": "#FBCFE2",
            "icon": draw_icon_circular_arrow,
            "rows": [
                ("Feature Projection", "f_proj = LayerNorm(GELU(W_f f + b_f)) in R^768"),
                ("Cross-Modal Gating", "g = sigma(W_g [h_[CLS] ; f_proj] + b_g)"),
                ("Residual Calibration", "H_tilde = LayerNorm(H + g * f_proj)"),
            ],
            "footer": "Out: Fused States H_tilde in R^(L x 768)",
        },
        {
            "num": "Stage 5",
            "title": "Rationale Masking",
            "subtitle": "Soft Mask m_a | Overlap Loss",
            "color": "#FF9500", "tint": "#FFF6E8", "border": "#FFE0B2",
            "icon": draw_icon_optimization,
            "rows": [
                ("Aspect Soft Mask", "m_a,i = sigma(w_a^T h_tilde_i + b_a) in [0, 1]"),
                ("Masked Span Pooling", "z_a = sum(m_a,i h_tilde_i) / sum(m_a,i)"),
                ("Orthogonal Overlap Loss", "L_ovl = sum_{a != b} (m_a . m_b) / (||m_a|| ||m_b||)"),
            ],
            "footer": "Out: Aligned z_a in R^768 (Zero Leakage)",
        },
        {
            "num": "Stage 6",
            "title": "3-Class Sentiment",
            "subtitle": "CORN Ordinal | Acc@0 & Macro-MAE",
            "color": "#FF5733", "tint": "#FFF0EC", "border": "#FFD1C4",
            "icon": draw_icon_chart_nodes,
            "rows": [
                ("CORN Conditional Ranks", "c_0 = I(y > Neg),  c_1 = I(y > Neu)"),
                ("Monotonic Probabilities", "P(Neg) = 1 - p_0, P(Neu) = p_0 - p_1, P(Pos) = p_1"),
                ("Multi-Task Objective", "L = L_CORN + lambda_1 L_rat + lambda_2 L_ovl"),
            ],
            "footer": "Order: Negative (0) < Neutral (1) < Positive (2)",
        },
    ]

    card_w = 4.30
    card_h = 4.25
    gap_x = 0.45
    start_x = 0.45

    card_y1 = 5.05
    top1 = card_y1 + card_h  # 9.30

    card_y2 = 0.45
    top2 = card_y2 + card_h  # 4.70

    # RENDER ROW 1
    for i, st in enumerate(stages_row1):
        x0 = start_x + i * (card_w + gap_x)
        draw_rounded_card(ax, x0, card_y1, card_w, card_h, bg="#FFFFFF", edgecolor="#E2E8F0",
                          lw=1.1, radius=0.22, shadow=True, zorder=3)

        st["icon"](ax, x0 + 0.44, top1 - 0.45, scale=0.85, zorder=6)

        # Title & Subtitle
        ax.text(x0 + 0.96, top1 - 0.35, st["title"], ha="left", va="center",
                fontsize=11.8, fontweight="bold", color="#0F172A", zorder=7)
        ax.text(x0 + 0.96, top1 - 0.62, st["subtitle"], ha="left", va="center",
                fontsize=9.0, fontweight="bold", color=st["color"], zorder=7)

        # Badge
        badge_w, badge_h = 0.78, 0.26
        bx = x0 + card_w - badge_w - 0.16
        by = top1 - 0.52
        ax.add_patch(FancyBboxPatch((bx, by), badge_w, badge_h, boxstyle="round,pad=0.0,rounding_size=0.08",
                                    facecolor=st["tint"], edgecolor=st["border"], lw=0.9, zorder=5))
        ax.text(bx + badge_w / 2, by + badge_h / 2, st["num"], ha="center", va="center",
                fontsize=8.5, fontweight="bold", color=st["color"], zorder=6)

        # Divider
        ax.plot([x0 + 0.16, x0 + card_w - 0.16], [top1 - 0.80, top1 - 0.80], color="#E2E8F0", lw=1.0, zorder=5)

        # 3 Bullet Items
        bullet_ys = [top1 - 1.25, top1 - 2.05, top1 - 2.85]
        for r_idx, (lbl, desc) in enumerate(st["rows"]):
            by_pos = bullet_ys[r_idx]
            ax.add_patch(Circle((x0 + 0.30, by_pos + 0.05), 0.040, facecolor=st["color"], edgecolor="none", zorder=6))
            ax.text(x0 + 0.44, by_pos + 0.05, lbl, ha="left", va="center",
                    fontsize=9.5, fontweight="bold", color="#0F172A", zorder=7)
            ax.text(x0 + 0.44, by_pos - 0.18, desc, ha="left", va="center",
                    fontsize=8.4, fontweight="normal", color="#475569", zorder=7)

        # Footer Badge
        foot_w = card_w - 0.32
        foot_h = 0.38
        fx = x0 + 0.16
        fy = card_y1 + 0.18
        ax.add_patch(FancyBboxPatch((fx, fy), foot_w, foot_h, boxstyle="round,pad=0.0,rounding_size=0.08",
                                    facecolor=st["tint"], edgecolor=st["border"], lw=0.9, zorder=5))
        ax.text(fx + foot_w / 2, fy + foot_h / 2, st["footer"], ha="center", va="center",
                fontsize=8.8, fontweight="bold", color="#1E293B", zorder=7)

        # Horizontal arrow Row 1
        if i < len(stages_row1) - 1:
            draw_step_arrow(ax, x0 + card_w + 0.04, x0 + card_w + gap_x - 0.04, card_y1 + card_h / 2,
                            color="#64748B", lw=2.2, head_w=0.10, head_h=0.07, zorder=4)

    # RENDER ROW 2
    for i, st in enumerate(stages_row2):
        x0 = start_x + i * (card_w + gap_x)
        draw_rounded_card(ax, x0, card_y2, card_w, card_h, bg="#FFFFFF", edgecolor="#E2E8F0",
                          lw=1.1, radius=0.22, shadow=True, zorder=3)

        st["icon"](ax, x0 + 0.44, top2 - 0.45, scale=0.85, zorder=6)

        # Title & Subtitle
        ax.text(x0 + 0.96, top2 - 0.35, st["title"], ha="left", va="center",
                fontsize=11.8, fontweight="bold", color="#0F172A", zorder=7)
        ax.text(x0 + 0.96, top2 - 0.62, st["subtitle"], ha="left", va="center",
                fontsize=9.0, fontweight="bold", color=st["color"], zorder=7)

        # Badge
        badge_w, badge_h = 0.78, 0.26
        bx = x0 + card_w - badge_w - 0.16
        by = top2 - 0.52
        ax.add_patch(FancyBboxPatch((bx, by), badge_w, badge_h, boxstyle="round,pad=0.0,rounding_size=0.08",
                                    facecolor=st["tint"], edgecolor=st["border"], lw=0.9, zorder=5))
        ax.text(bx + badge_w / 2, by + badge_h / 2, st["num"], ha="center", va="center",
                fontsize=8.5, fontweight="bold", color=st["color"], zorder=6)

        # Divider
        ax.plot([x0 + 0.16, x0 + card_w - 0.16], [top2 - 0.80, top2 - 0.80], color="#E2E8F0", lw=1.0, zorder=5)

        # 3 Bullet Items
        bullet_ys = [top2 - 1.25, top2 - 2.05, top2 - 2.85]
        for r_idx, (lbl, desc) in enumerate(st["rows"]):
            by_pos = bullet_ys[r_idx]
            ax.add_patch(Circle((x0 + 0.30, by_pos + 0.05), 0.040, facecolor=st["color"], edgecolor="none", zorder=6))
            ax.text(x0 + 0.44, by_pos + 0.05, lbl, ha="left", va="center",
                    fontsize=9.5, fontweight="bold", color="#0F172A", zorder=7)
            ax.text(x0 + 0.44, by_pos - 0.18, desc, ha="left", va="center",
                    fontsize=8.4, fontweight="normal", color="#475569", zorder=7)

        # Footer Badge
        foot_w = card_w - 0.32
        foot_h = 0.38
        fx = x0 + 0.16
        fy = card_y2 + 0.18
        ax.add_patch(FancyBboxPatch((fx, fy), foot_w, foot_h, boxstyle="round,pad=0.0,rounding_size=0.08",
                                    facecolor=st["tint"], edgecolor=st["border"], lw=0.9, zorder=5))
        ax.text(fx + foot_w / 2, fy + foot_h / 2, st["footer"], ha="center", va="center",
                fontsize=8.8, fontweight="bold", color="#1E293B", zorder=7)

        # Horizontal arrow Row 2
        if i < len(stages_row2) - 1:
            draw_step_arrow(ax, x0 + card_w + 0.04, x0 + card_w + gap_x - 0.04, card_y2 + card_h / 2,
                            color="#64748B", lw=2.2, head_w=0.10, head_h=0.07, zorder=4)

    # SERPENTINE CONNECTOR BUS:
    bus_y = 4.875
    s3_cx = start_x + 2 * (card_w + gap_x) + card_w / 2  # 12.10
    s2_cx = start_x + 1 * (card_w + gap_x) + card_w / 2  # 7.35
    s4_cy = card_y2 + card_h / 2                         # 2.575
    loop_x = 0.30

    # Bus description badge position
    bus_badge_w = 3.80
    bus_badge_h = 0.25
    bb_cx = 4.80
    bb_x = bb_cx - bus_badge_w / 2  # 2.90
    bb_y = bus_y - bus_badge_h / 2

    # Drop from Stage 3
    ax.plot([s3_cx, s3_cx], [card_y1, bus_y], color="#1DB096", lw=2.2, solid_capstyle="round", zorder=4)
    ax.add_patch(Circle((s3_cx, card_y1), 0.045, facecolor="#1DB096", edgecolor="none", zorder=5))
    ax.text(s3_cx + 0.12, (card_y1 + bus_y) / 2, "H in R^(L x 768)", ha="left", va="center",
            fontsize=7.8, fontweight="bold", color="#0F766E", zorder=6)

    # Drop from Stage 2 (Feature shortcut)
    ax.plot([s2_cx, s2_cx], [card_y1, bus_y], color="#2997FF", lw=2.2, solid_capstyle="round", zorder=4)
    ax.add_patch(Circle((s2_cx, card_y1), 0.045, facecolor="#2997FF", edgecolor="none", zorder=5))
    ax.text(s2_cx + 0.12, (card_y1 + bus_y) / 2, "f in R^20", ha="left", va="center",
            fontsize=7.8, fontweight="bold", color="#0284C7", zorder=6)

    # Segment 1: from s3_cx (12.10) to badge right edge (bb_x + bus_badge_w = 6.70)
    ax.plot([bb_x + bus_badge_w, s3_cx], [bus_y, bus_y], color="#64748B", lw=2.2, solid_capstyle="round", zorder=4)
    ax.add_patch(Circle((s3_cx, bus_y), 0.045, facecolor="#1DB096", edgecolor="none", zorder=5))
    ax.add_patch(Circle((s2_cx, bus_y), 0.045, facecolor="#2997FF", edgecolor="none", zorder=5))

    # Bus badge (hub) - rounded card with subtle shadow and border
    draw_rounded_card(ax, bb_x, bb_y, bus_badge_w, bus_badge_h, bg="#FFFFFF", edgecolor="#94A3B8", lw=1.1, radius=0.07, shadow=True, zorder=5)
    ax.text(bb_cx, bus_y, "Cross-Modal Feed: H (768d) + f (20d)",
            ha="center", va="center", fontsize=8.0, fontweight="bold", color="#1E293B", zorder=7)

    # Segment 2: from badge left edge (bb_x = 2.90) to loop_x (0.30)
    ax.plot([loop_x, bb_x], [bus_y, bus_y], color="#64748B", lw=2.2, solid_capstyle="round", zorder=4)

    # Down line along left margin
    ax.plot([loop_x, loop_x], [bus_y, s4_cy], color="#64748B", lw=2.2, solid_capstyle="round", zorder=4)

    # Step arrow entering Stage 4 left edge (x=0.45, y=2.575)
    draw_step_arrow(ax, loop_x, start_x, s4_cy, color="#64748B", lw=2.2, head_w=0.08, head_h=0.065, zorder=4)

    # RIGHT COLUMN: Full-Height (rx=14.80, rw=4.15, ry=0.45, rh=8.85)
    rx = 14.80
    rw = 4.15
    ry = 0.45
    rh = 8.85
    rcx = rx + rw / 2
    top_r = ry + rh  # 9.30

    draw_rounded_card(ax, rx, ry, rw, rh, bg="#FFFFFF", edgecolor="#DCE0E8", lw=1.2, radius=0.24, shadow=True, zorder=3)

    # Upper section: 6-Aspect Adapters
    ax.text(rcx, top_r - 0.38, "6-Aspect Adapters", ha="center", va="center",
            fontsize=13.8, fontweight="bold", color="#0F172A", zorder=7)
    ax.text(rcx, top_r - 0.65, "Task-Specific Adapter Heads", ha="center", va="center",
            fontsize=9.2, fontweight="bold", color="#0284C7", zorder=7)
    ax.plot([rx + 0.22, rx + rw - 0.22], [top_r - 0.86, top_r - 0.86], color="#E2E8F0", lw=1.0, zorder=5)

    aspects = ["Service", "Facility", "Exper.", "Amenity", "Loyalty", "Brand."]
    pill_w = 0.50
    pill_gap = 0.12
    total_pill_w = 6 * pill_w + 5 * pill_gap
    start_pill_x = rcx - total_pill_w / 2
    pill_h = 1.60
    pill_y = top_r - 1.06 - pill_h
    centers = []

    for a_i, lbl in enumerate(aspects):
        bx = start_pill_x + a_i * (pill_w + pill_gap)
        bcx = bx + pill_w / 2
        centers.append(bcx)
        ax.add_patch(FancyBboxPatch((bx, pill_y), pill_w, pill_h, boxstyle="round,pad=0.0,rounding_size=0.10",
                                    facecolor="#0099C6", edgecolor="none", zorder=5))
        for d_i in range(3):
            ax.add_patch(Circle((bcx, pill_y + pill_h - 0.14 - d_i * 0.08), 0.022,
                                facecolor="#FFFFFF", edgecolor="none", zorder=6))
        ax.text(bcx, pill_y + (pill_h - 0.38) / 2, lbl, ha="center", va="center", rotation=90,
                fontsize=9.6, fontweight="bold", color="#FFFFFF", zorder=7)

    # Collector tree arrow pointing down
    y_bar = pill_y - 0.22
    y_tip = pill_y - 0.56
    for bcx in centers:
        ax.plot([bcx, bcx], [pill_y, y_bar], color="#0099C6", lw=1.6, zorder=5)
    ax.plot([centers[0], centers[-1]], [y_bar, y_bar], color="#0099C6", lw=2.0, solid_capstyle="round", zorder=5)
    draw_down_arrow(ax, rcx, y_bar, y_tip, color="#0099C6", lw=2.2, head_w=0.10, head_h=0.12, zorder=5)

    # Lower section: CRAM-ABSA Output card
    oc_x = rx + 0.16
    oc_w = rw - 0.32
    oc_top = y_tip - 0.08  # ~ 5.38
    oc_y = ry + 0.16
    oc_h = oc_top - oc_y   # ~ 4.77

    draw_rounded_card(ax, oc_x, oc_y, oc_w, oc_h, bg="#FFFFFF", edgecolor="#BAE6FD", lw=1.3, radius=0.18, shadow=False, zorder=5)

    ax.text(rcx, oc_top - 0.32, "CRAM-ABSA Output & Synthesis", ha="center", va="center",
            fontsize=12.2, fontweight="bold", color="#0F172A", zorder=7)
    ax.text(rcx, oc_top - 0.56, "Evidence-Grounded Multi-Task Predictions", ha="center", va="center",
            fontsize=8.8, fontweight="bold", color="#0284C7", zorder=7)
    ax.plot([oc_x + 0.18, oc_x + oc_w - 0.18], [oc_top - 0.74, oc_top - 0.74], color="#E2E8F0", lw=1.0, zorder=6)

    # 4 Output Sections (distribute evenly across available height)
    out_sections = [
        ("• Aspect State: 6 Mentioned Binary Classes", "Aspect indicators y_a in {0, 1}^6 via calibrated sigmoid", "#0066CC", "#F0F9FF", "#BAE6FD"),
        ("• Sentiment: 3-Class (Positive / Neutral / Negative)", "CORN ordinal prob: P(Neg)=1-p_0, P(Neu)=p_0-p_1, P(Pos)=p_1", "#1E293B", "#F8FAFC", "#E2E8F0"),
        ("• Grounded Rationale Spans (Negation-Aware)", "Orthogonal soft attention m_a (inter-aspect overlap <= 0.24%)", "#475569", "#F8FAFC", "#E2E8F0"),
        ("• Evidence-Weighted Consensus: S_hat_a in [-1, +1]", "Continuous consensus polarity: S_hat_a = P(Pos) - P(Neg)", "#0D9488", "#F0FDF4", "#BBF7D0"),
    ]

    sec_w = oc_w - 0.24
    sec_h = 0.76
    sec_x = oc_x + 0.12
    avail_h = (oc_top - 0.82) - (oc_y + 0.14)
    sec_gap = (avail_h - 4 * sec_h) / 3

    for s_idx, (title, desc, txt_col, bg_col, bdr_col) in enumerate(out_sections):
        sy = (oc_top - 0.82) - sec_h - s_idx * (sec_h + sec_gap)
        draw_rounded_card(ax, sec_x, sy, sec_w, sec_h, bg=bg_col, edgecolor=bdr_col, lw=0.9, radius=0.10, shadow=False, zorder=6)
        ax.text(sec_x + 0.16, sy + sec_h - 0.26, title, ha="left", va="center",
                fontsize=9.2, fontweight="bold", color=txt_col, zorder=7)
        ax.text(sec_x + 0.16, sy + 0.24, desc, ha="left", va="center",
                fontsize=8.1, fontweight="normal", color="#475569", zorder=7)

    # Arrow from Stage 6 into Right Box (at y=2.575)
    s6_rx = start_x + 2 * (card_w + gap_x) + card_w  # 14.25
    draw_step_arrow(ax, s6_rx + 0.04, oc_x - 0.02, s4_cy, color="#64748B", lw=2.2, head_w=0.10, head_h=0.07, zorder=7)
    ax.text((s6_rx + oc_x) / 2, s4_cy + 0.14, "z_a, c_k", ha="center", va="center",
            fontsize=7.8, fontweight="bold", color="#64748B", zorder=7)

    out_p = "docs/main_paper/chart_img/412_abstract_architecture_of_proposed_method.png"
    plt.savefig(out_p, dpi=300, bbox_inches="tight", pad_inches=0.05, facecolor="white")
    plt.close(fig)
    print(f"Generated {out_p} successfully.")

build_figure_412()

Generated detailed data/chart_img/412_abstract_architecture_of_proposed_method.png and synced to docs/main_paper/chart_img/412_abstract_architecture_of_proposed_method.png successfully.


### 4.13 Đồ thị So sánh Benchmark Định lượng (Quantitative Benchmark Comparison)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("docs/main_paper/chart_img", exist_ok=True)

# Publication-grade styling for Springer LNCS/journal
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 10.5,
    "axes.titlesize": 11.5,
    "axes.labelsize": 11,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10,
    "legend.fontsize": 9.8,
    "figure.dpi": 350,
    "savefig.dpi": 350,
    "axes.linewidth": 0.9,
})

def generate_figure_413():
    metrics = ["Acc@0 (%)", "6-Aspect Macro-F1 (%)", "Rationale F1 (%)"]
    models = [
        ("CRAM-ABSA (Ours)",   [92.39, 72.55, 70.84], "#0072B2", None),
        ("BERT-Base [1]",      [92.29, 52.55, 50.66], "#D55E00", "//"),
        ("BM25-CNN [4]",       [91.69, 44.22, 32.65], "#009E73", ".."),
        ("WSD-BiLSTM-GCN [9]", [92.59, 52.12, 9.77],  "#CC79A7", "xx"),
    ]

    x = np.arange(len(metrics))
    n_models = len(models)
    bar_width = 0.19
    offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * (bar_width + 0.012)

    fig, ax = plt.subplots(figsize=(7.6, 4.1))

    for i, (label, vals, color, hatch) in enumerate(models):
        bars = ax.bar(
            x + offsets[i],
            vals,
            width=bar_width,
            label=label,
            color=color,
            edgecolor="#222222",
            linewidth=0.7,
            hatch=hatch,
            alpha=0.92,
            zorder=3,
        )
        for bar, val in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                bar.get_height() + 1.4,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=8.6,
                fontweight="bold" if i == 0 else "normal",
                color="#111111",
            )

    ax.set_ylabel("Score (%) — Higher is Better ($\\uparrow$)", fontweight="bold", labelpad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontweight="bold")
    ax.set_ylim(0, 106)
    ax.set_yticks(np.arange(0, 101, 20))
    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, color="#CCCCCC", alpha=0.8, zorder=0)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", length=4)
    ax.tick_params(axis="y", length=4)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.13),
        ncol=4,
        frameon=True,
        facecolor="white",
        edgecolor="#DDDDDD",
        columnspacing=1.1,
        handletextpad=0.5,
    )

    plt.tight_layout()
    out_path = "docs/main_paper/chart_img/413_benchmark_comparison.png"
    fig.savefig(out_path, bbox_inches="tight", dpi=350, facecolor="white")
    plt.close(fig)
    print(f"Generated Springer-conference Quality Metrics chart at {out_path} successfully.")

generate_figure_413()


## 5. Feature Engineering


### 5.1 Tạo 20 Features


In [24]:
try:
    from nltk.corpus import stopwords
    NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

HEDGING_MODIFIERS = {
    "okay", "ok", "decent", "average", "fine", "fair", "moderate", "standard",
    "acceptable", "passable", "normal", "typical", "ordinary", "somewhat", "slightly",
    "maybe", "perhaps", "quite", "rather", "relatively", "adequate", "mediocre",
    "reasonable", "reasonably"
}

SAFE_STOPWORDS = NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "basic", "boring", "crowded", "weak", "fault"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably"
}

CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0
SENTIMENT_VAL_MAP = {"Positive": 1.0, "Neutral": 0.0, "Negative": -1.0}

ASPECT_KEYWORDS = {
    "Service": {"staff", "service", "reception", "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful", "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping", "host", "crew"},
    "Facility": {"room", "bed", "bathroom", "shower", "toilet", "air", "ac", "conditioning", "clean", "cleanliness", "noise", "noisy", "soundproof", "view", "balcony", "floor", "elevator", "lift", "pillow", "mattress", "furniture", "decor", "water", "carpet"},
    "Experience": {"stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable", "time", "comfort", "comfortable", "quiet", "peaceful", "feeling"},
    "Amenity": {"breakfast", "buffet", "pool", "swimming", "spa", "gym", "fitness", "wifi", "internet", "restaurant", "bar", "food", "drink", "coffee", "tea", "lounge", "shuttle", "parking", "massage", "rooftop"},
    "Loyalty": {"back", "return", "again", "recommend", "recommended", "definitely", "choose", "future", "next", "visit", "repeat", "choice", "staying"},
    "Branding": {"star", "stars", "luxury", "brand", "hotel", "resort", "standard", "reputation", "class", "five", "four", "chain", "management", "value", "price", "worth", "quality", "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton", "intercontinental", "sheraton"}
}

def nltk_negation_aware_preprocess(text: str):
    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [t for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens

def preprocess_text_paper1(text: str):
    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    return clean_review_text, " ".join(clean_tokens)

def infer_aspect_sentiment_from_text_nltk(raw_text: str, target_asp: str) -> float:
    sentences = [s.strip() for s in re.split(r"[.!?]+", str(raw_text)) if s.strip()]
    kw_set = ASPECT_KEYWORDS[target_asp]
    scores = []
    for sent in sentences:
        raw_toks, clean_toks = nltk_negation_aware_preprocess(sent)
        if any(t in kw_set for t in clean_toks):
            pos_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
            neg_hits = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)
            for idx_t, tok in enumerate(raw_toks[:-1]):
                if tok in {"not", "never", "no", "hardly"}:
                    nxt = raw_toks[idx_t + 1]
                    if nxt in OPINION_LEXICON_POS:
                        pos_hits = max(0, pos_hits - 1)
                        neg_hits += 1
                    elif nxt in OPINION_LEXICON_NEG:
                        neg_hits = max(0, neg_hits - 1)
                        pos_hits += 1
            scores.append((pos_hits - neg_hits) / (pos_hits + neg_hits) if (pos_hits + neg_hits) > 0 else 0.0)
    return float(round(sum(scores) / len(scores), 4)) if scores else 0.0

df_features = pd.DataFrame({"review_id": df_reviews["review_id"]})
ann_by_review = defaultdict(list)
for rec in df_annotations[["review_id", "aspect", "sentiment", "start_char", "end_char"]].to_dict("records"):
    ann_by_review[int(rec["review_id"])].append(rec)

review_cache = {}
for row in df_reviews[["review_id", "review_text", "char_length"]].to_dict("records"):
    rid = int(row["review_id"])
    raw_txt = str(row["review_text"])
    raw_toks, clean_toks = nltk_negation_aware_preprocess(raw_txt)
    recs = ann_by_review.get(rid, [])
    inferred = {asp: infer_aspect_sentiment_from_text_nltk(raw_txt, asp) for asp in CANONICAL_ASPECTS} if not recs else {}
    review_cache[rid] = {
        "raw_text": raw_txt,
        "char_len": int(row["char_length"]),
        "raw_tokens": raw_toks,
        "clean_tokens": clean_toks,
        "records": recs,
        "inferred": inferred
    }

def compute_aspect_polarity(rid: int, target_asp: str) -> float:
    c = review_cache[rid]
    recs = [r for r in c["records"] if r["aspect"] == target_asp]
    if not recs:
        return c["inferred"].get(target_asp, 0.0) if not c["records"] else 0.0
    w_sum = sum(max(1, int(r["end_char"]) - int(r["start_char"])) * SENTIMENT_VAL_MAP.get(r["sentiment"], 0.0) for r in recs)
    t_len = sum(max(1, int(r["end_char"]) - int(r["start_char"])) for r in recs)
    return float(round(w_sum / t_len, 4)) if t_len > 0 else 0.0

print(f"Đã khởi tạo df_features và bộ nhớ đệm tiền xử lý cho {len(df_features):,} reviews.")

Đã khởi tạo df_features và bộ nhớ đệm tiền xử lý cho 9,990 reviews.


#### Feature 01: `feat_aspect_count`
Đếm số lượng khía cạnh phân biệt xuất hiện trong bài đánh giá (từ 0 đến 6). Đặc trưng này phản ánh độ rộng thông tin để mô hình ABSA điều chỉnh trọng số chú ý đa khía cạnh.


In [25]:
def calc_feat_aspect_count(rid):
    c = review_cache[rid]
    if c["records"]:
        return len({r["aspect"] for r in c["records"]})
    return sum(1 for asp, sc in c["inferred"].items() if abs(sc) > 0 or any(t in ASPECT_KEYWORDS[asp] for t in c["clean_tokens"]))

df_features["feat_aspect_count"] = [calc_feat_aspect_count(int(r)) for r in df_features["review_id"]]
print(f"feat_aspect_count -> mean: {df_features['feat_aspect_count'].mean():.4f}, min: {df_features['feat_aspect_count'].min()}, max: {df_features['feat_aspect_count'].max()}")

feat_aspect_count -> mean: 3.3973, min: 0, max: 6


#### Feature 02: `feat_dominant_aspect`
Mã hóa ID (1..6, hoặc 0 nếu không có) của khía cạnh có tần suất xuất hiện nhiều nhất trong bài đánh giá. Giúp định hướng khía cạnh trọng tâm quyết định nhãn cảm xúc 3 lớp (Negative, Neutral, Positive).


In [26]:
def calc_feat_dominant_aspect(rid):
    c = review_cache[rid]
    if c["records"]:
        counts = Counter(r["aspect"] for r in c["records"])
        dom = counts.most_common(1)[0][0]
        return int(ASPECT_ID_MAP.get(dom, 0))
    active = [asp for asp, sc in c["inferred"].items() if abs(sc) > 0 or any(t in ASPECT_KEYWORDS[asp] for t in c["clean_tokens"])]
    return int(ASPECT_ID_MAP.get(active[0], 0)) if active else 0

df_features["feat_dominant_aspect"] = [calc_feat_dominant_aspect(int(r)) for r in df_features["review_id"]]
print(f"feat_dominant_aspect -> mode: {int(df_features['feat_dominant_aspect'].mode().iloc[0])}, unique IDs: {sorted(df_features['feat_dominant_aspect'].unique().tolist())}")

feat_dominant_aspect -> mode: 1, unique IDs: [0, 1, 2, 3, 4, 5, 6]


#### Feature 03: `feat_net_sentiment_score`
Hiệu số chuẩn hóa giữa số lượng đoạn bằng chứng Positive và Negative chia cho tổng số đoạn bằng chứng trong `[-1.0, 1.0]`. Cung cấp tín hiệu phân cực tổng hợp trực tiếp cho bộ phân loại cảm xúc 3 lớp.


In [27]:
def calc_feat_net_sentiment_score(rid):
    c = review_cache[rid]
    recs = c["records"]
    if recs:
        pos_c = sum(1 for r in recs if r["sentiment"] == "Positive")
        neg_c = sum(1 for r in recs if r["sentiment"] == "Negative")
        return float(round((pos_c - neg_c) / len(recs), 4))
    vals = [v for v in c["inferred"].values() if abs(v) > 0]
    return float(round(sum(vals) / len(vals), 4)) if vals else 0.0

df_features["feat_net_sentiment_score"] = [calc_feat_net_sentiment_score(int(r)) for r in df_features["review_id"]]
print(f"feat_net_sentiment_score -> mean: {df_features['feat_net_sentiment_score'].mean():.4f}, min: {df_features['feat_net_sentiment_score'].min():.4f}, max: {df_features['feat_net_sentiment_score'].max():.4f}")

feat_net_sentiment_score -> mean: 0.7741, min: -1.0000, max: 1.0000


#### Feature 04: `feat_sentiment_conflict`
Cờ nhị phân `{0, 1}` bật lên 1 khi bài đánh giá vừa có khía cạnh Positive vừa có khía cạnh Negative. Hỗ trợ nhận diện các bài đánh giá đa chiều dễ rơi vào lớp Neutral hoặc có xung đột ngữ nghĩa.


In [28]:
def calc_feat_sentiment_conflict(rid):
    c = review_cache[rid]
    recs = c["records"]
    if recs:
        has_pos = any(r["sentiment"] == "Positive" for r in recs)
        has_neg = any(r["sentiment"] == "Negative" for r in recs)
        return 1 if (has_pos and has_neg) else 0
    vals = list(c["inferred"].values())
    return 1 if (any(v > 0 for v in vals) and any(v < 0 for v in vals)) else 0

df_features["feat_sentiment_conflict"] = [calc_feat_sentiment_conflict(int(r)) for r in df_features["review_id"]]
print(f"feat_sentiment_conflict -> conflict rate: {df_features['feat_sentiment_conflict'].mean()*100:.2f}% ({df_features['feat_sentiment_conflict'].sum():,} reviews)")

feat_sentiment_conflict -> conflict rate: 16.66% (1,664 reviews)


#### Feature 05: `feat_evidence_density`
Tỷ lệ tổng số ký tự thuộc các đoạn bằng chứng (rationale spans) trên tổng độ dài ký tự bài đánh giá trong `[0.0, 1.0]`. Đo lường mật độ thông tin hữu ích dùng làm trọng số tin cậy khi tổng hợp đa đánh giá.


In [29]:
def calc_feat_evidence_density(rid):
    c = review_cache[rid]
    if c["records"] and c["char_len"] > 0:
        span_chars = sum(max(0, int(r["end_char"]) - int(r["start_char"])) for r in c["records"])
        return float(round(min(span_chars / c["char_len"], 1.0), 4))
    return float(round(min(len(c["clean_tokens"]) / max(len(c["raw_tokens"]), 1), 1.0), 4))

df_features["feat_evidence_density"] = [calc_feat_evidence_density(int(r)) for r in df_features["review_id"]]
print(f"feat_evidence_density -> mean: {df_features['feat_evidence_density'].mean():.4f}, min: {df_features['feat_evidence_density'].min():.4f}, max: {df_features['feat_evidence_density'].max():.4f}")

feat_evidence_density -> mean: 0.7954, min: 0.0419, max: 1.0000


#### Feature 06: `feat_avg_rationale_len`
Độ dài ký tự trung bình của các đoạn bằng chứng trong bài đánh giá. Phản ánh mức độ chi tiết của lập luận khi khách hàng khen hoặc chê từng khía cạnh.


In [30]:
def calc_feat_avg_rationale_len(rid):
    c = review_cache[rid]
    if c["records"]:
        lengths = [max(0, int(r["end_char"]) - int(r["start_char"])) for r in c["records"]]
        return float(round(sum(lengths) / len(lengths), 2))
    return float(round(c["char_len"] / max(int(df_features.loc[rid - 1, "feat_aspect_count"]), 1), 2))

df_features["feat_avg_rationale_len"] = [calc_feat_avg_rationale_len(int(r)) for r in df_features["review_id"]]
print(f"feat_avg_rationale_len -> mean: {df_features['feat_avg_rationale_len'].mean():.2f}, min: {df_features['feat_avg_rationale_len'].min():.2f}, max: {df_features['feat_avg_rationale_len'].max():.2f}")

feat_avg_rationale_len -> mean: 66.63, min: 16.00, max: 394.50


#### Feature 07: `feat_has_cross_overlap`
Cờ nhị phân `{0, 1}` kiểm tra sự chồng lấn tọa độ ký tự giữa hai khía cạnh khác nhau trong cùng một bài đánh giá. Cung cấp tín hiệu điều phạt chồng lấn (Overlap Penalty) cho cơ chế Rationale Masking.


In [31]:
def calc_feat_has_cross_overlap(rid):
    recs = review_cache[rid]["records"]
    n = len(recs)
    for i in range(n):
        for j in range(i + 1, n):
            if recs[i]["aspect"] != recs[j]["aspect"]:
                if not (recs[i]["end_char"] <= recs[j]["start_char"] or recs[i]["start_char"] >= recs[j]["end_char"]):
                    return 1
    return 0

df_features["feat_has_cross_overlap"] = [calc_feat_has_cross_overlap(int(r)) for r in df_features["review_id"]]
print(f"feat_has_cross_overlap -> overlapping reviews: {df_features['feat_has_cross_overlap'].sum()} / {len(df_features):,}")

feat_has_cross_overlap -> overlapping reviews: 6 / 9,990


#### Feature 08: `feat_service_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Service` trong `[-1.0, 1.0]`. Biểu diễn trực tiếp thái độ của khách hàng đối với chất lượng phục vụ và nhân viên khách sạn.


In [32]:
df_features["feat_service_sentiment"] = [compute_aspect_polarity(int(r), "Service") for r in df_features["review_id"]]
print(f"feat_service_sentiment -> mean: {df_features['feat_service_sentiment'].mean():.4f}, non-zero: {(df_features['feat_service_sentiment'] != 0).sum():,}")

feat_service_sentiment -> mean: 0.7559, non-zero: 9,044


#### Feature 09: `feat_facility_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Facility` trong `[-1.0, 1.0]`. Phản ánh đánh giá về phòng ốc, giường ngủ, cách âm và trang thiết bị nội thất.


In [33]:
df_features["feat_facility_sentiment"] = [compute_aspect_polarity(int(r), "Facility") for r in df_features["review_id"]]
print(f"feat_facility_sentiment -> mean: {df_features['feat_facility_sentiment'].mean():.4f}, non-zero: {(df_features['feat_facility_sentiment'] != 0).sum():,}")

feat_facility_sentiment -> mean: 0.5358, non-zero: 7,200


#### Feature 10: `feat_experience_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Experience` trong `[-1.0, 1.0]`. Đo lường mức độ hài lòng tổng thể về kỳ nghỉ, không gian và sự thoải mái khi lưu trú.


In [34]:
df_features["feat_experience_sentiment"] = [compute_aspect_polarity(int(r), "Experience") for r in df_features["review_id"]]
print(f"feat_experience_sentiment -> mean: {df_features['feat_experience_sentiment'].mean():.4f}, non-zero: {(df_features['feat_experience_sentiment'] != 0).sum():,}")

feat_experience_sentiment -> mean: 0.4602, non-zero: 5,908


#### Feature 11: `feat_amenity_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Amenity` trong `[-1.0, 1.0]`. Ghi nhận cảm xúc 3 lớp về các tiện ích đi kèm như bữa sáng, hồ bơi, quầy bar, spa và wifi.


In [35]:
df_features["feat_amenity_sentiment"] = [compute_aspect_polarity(int(r), "Amenity") for r in df_features["review_id"]]
print(f"feat_amenity_sentiment -> mean: {df_features['feat_amenity_sentiment'].mean():.4f}, non-zero: {(df_features['feat_amenity_sentiment'] != 0).sum():,}")

feat_amenity_sentiment -> mean: 0.4850, non-zero: 5,457


#### Feature 12: `feat_loyalty_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Loyalty` trong `[-1.0, 1.0]`. Nắm bắt ý định quay lại hoặc sẵn sàng giới thiệu khách sạn của khách hàng.


In [36]:
df_features["feat_loyalty_sentiment"] = [compute_aspect_polarity(int(r), "Loyalty") for r in df_features["review_id"]]
print(f"feat_loyalty_sentiment -> mean: {df_features['feat_loyalty_sentiment'].mean():.4f}, non-zero: {(df_features['feat_loyalty_sentiment'] != 0).sum():,}")

feat_loyalty_sentiment -> mean: 0.3532, non-zero: 3,928


#### Feature 13: `feat_branding_sentiment`
Điểm phân cực cảm xúc có trọng số độ dài đoạn bằng chứng của khía cạnh `Branding` trong `[-1.0, 1.0]`. Đánh giá nhận thức về đẳng cấp thương hiệu, tiêu chuẩn khách sạn và tương xứng giá trị.


In [37]:
df_features["feat_branding_sentiment"] = [compute_aspect_polarity(int(r), "Branding") for r in df_features["review_id"]]
print(f"feat_branding_sentiment -> mean: {df_features['feat_branding_sentiment'].mean():.4f}, non-zero: {(df_features['feat_branding_sentiment'] != 0).sum():,}")

feat_branding_sentiment -> mean: 0.0805, non-zero: 1,233


#### Feature 14: `feat_aspect_coverage_ratio`
Tỷ lệ bao phủ khía cạnh tính bằng `min(feat_aspect_count / 6.0, 1.0)` trong `[0.0, 1.0]`. Chuẩn hóa số lượng khía cạnh được đề cập về thang đo đơn vị cho tầng Feature Fusion.


In [38]:
df_features["feat_aspect_coverage_ratio"] = (df_features["feat_aspect_count"] / 6.0).clip(0.0, 1.0).round(4)
print(f"feat_aspect_coverage_ratio -> mean: {df_features['feat_aspect_coverage_ratio'].mean():.4f}, min: {df_features['feat_aspect_coverage_ratio'].min():.4f}, max: {df_features['feat_aspect_coverage_ratio'].max():.4f}")

feat_aspect_coverage_ratio -> mean: 0.5662, min: 0.0000, max: 1.0000


#### Feature 15: `feat_neutral_hedging_density`
Mật độ từ ngữ trung tính, rào đón và tương phản (`HEDGING_MODIFIERS | {'but', 'however', 'although', 'though', 'yet', 'despite'}`) trên tổng số từ gốc trong `[0.0, 1.0]`. Đặc trưng then chốt giúp phân biệt lớp `Neutral` với hai cực `Positive` và `Negative`.


In [39]:
HEDGING_CONTRAST_SET = HEDGING_MODIFIERS | {"but", "however", "although", "though", "yet", "despite"}

def calc_feat_neutral_hedging_density(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    if not raw_toks:
        return 0.0
    hits = sum(1 for t in raw_toks if t in HEDGING_CONTRAST_SET)
    return float(round(min(hits / len(raw_toks), 1.0), 4))

df_features["feat_neutral_hedging_density"] = [calc_feat_neutral_hedging_density(int(r)) for r in df_features["review_id"]]
print(f"feat_neutral_hedging_density -> mean: {df_features['feat_neutral_hedging_density'].mean():.4f}, max: {df_features['feat_neutral_hedging_density'].max():.4f}")

feat_neutral_hedging_density -> mean: 0.0109, max: 0.1538


#### Feature 16: `feat_consensus_strength`
Độ mạnh đồng thuận cảm xúc nội tại tính bằng `abs(feat_net_sentiment_score)` trong `[0.0, 1.0]`. Giá trị gần 1 biểu thị sự nhất quán một chiều (Positive hoặc Negative), giá trị gần 0 báo hiệu trạng thái cân bằng/Neutral.


In [40]:
df_features["feat_consensus_strength"] = df_features["feat_net_sentiment_score"].abs().round(4)
print(f"feat_consensus_strength -> mean: {df_features['feat_consensus_strength'].mean():.4f}, min: {df_features['feat_consensus_strength'].min():.4f}, max: {df_features['feat_consensus_strength'].max():.4f}")

feat_consensus_strength -> mean: 0.8602, min: 0.0000, max: 1.0000


#### Feature 17: `feat_lexical_polarity_intensity`
Cường độ phân cực từ vựng có xét đảo chiều phủ định và khuếch đại trạng từ (`INTENSIFIERS`) trong `[-1.0, 1.0]`. Bổ sung tín hiệu cảm xúc mức từ vựng độc lập với nhãn chú thích thủ công.


In [41]:
def calc_feat_lexical_polarity_intensity(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    pos_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
    neg_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)
    intens_cnt = sum(1 for t in raw_toks if t in INTENSIFIERS)
    for idx_t, tok in enumerate(raw_toks[:-1]):
        if tok in {"not", "never", "no", "hardly", "neither", "nor", "without"}:
            nxt = raw_toks[idx_t + 1]
            if nxt in OPINION_LEXICON_POS:
                pos_cnt = max(0, pos_cnt - 1)
                neg_cnt += 1
            elif nxt in OPINION_LEXICON_NEG:
                neg_cnt = max(0, neg_cnt - 1)
                pos_cnt += 1
    base_pol = (pos_cnt - neg_cnt) / max(pos_cnt + neg_cnt, 1) if (pos_cnt + neg_cnt) > 0 else 0.0
    intens_boost = min(0.35, 0.12 * intens_cnt) * (1.0 if base_pol >= 0 else -1.0) if base_pol != 0 else 0.0
    return float(round(np.clip(base_pol + intens_boost, -1.0, 1.0), 4))

df_features["feat_lexical_polarity_intensity"] = [calc_feat_lexical_polarity_intensity(int(r)) for r in df_features["review_id"]]
print(f"feat_lexical_polarity_intensity -> mean: {df_features['feat_lexical_polarity_intensity'].mean():.4f}, min: {df_features['feat_lexical_polarity_intensity'].min():.4f}, max: {df_features['feat_lexical_polarity_intensity'].max():.4f}")

feat_lexical_polarity_intensity -> mean: 0.8031, min: -1.0000, max: 1.0000


#### Feature 18: `feat_nltk_clean_token_ratio`
Tỷ lệ số lượng từ thực chất (sau khi lọc stopwords nhưng bảo toàn từ phủ định) trên tổng số từ gốc trong `[0.0, 1.0]`. Đánh giá độ cô đọng ngữ nghĩa của văn bản đầu vào theo quy trình tiền xử lý NLTK.


In [42]:
def calc_feat_nltk_clean_token_ratio(rid):
    c = review_cache[rid]
    return float(round(min(len(c["clean_tokens"]) / max(len(c["raw_tokens"]), 1), 1.0), 4))

df_features["feat_nltk_clean_token_ratio"] = [calc_feat_nltk_clean_token_ratio(int(r)) for r in df_features["review_id"]]
print(f"feat_nltk_clean_token_ratio -> mean: {df_features['feat_nltk_clean_token_ratio'].mean():.4f}, min: {df_features['feat_nltk_clean_token_ratio'].min():.4f}, max: {df_features['feat_nltk_clean_token_ratio'].max():.4f}")

feat_nltk_clean_token_ratio -> mean: 0.5466, min: 0.3333, max: 0.9048


#### Feature 19: `feat_nltk_opinion_negation_density`
Mật độ các từ mang sắc thái quan điểm (Positive/Negative), từ phủ định và trạng từ nhấn mạnh trên tổng số từ gốc trong `[0.0, 1.0]`. Đo lường mức độ giàu biểu cảm của câu phục vụ phân tích cảm xúc 3 lớp.


In [43]:
def calc_feat_nltk_opinion_negation_density(rid):
    raw_toks = review_cache[rid]["raw_tokens"]
    if not raw_toks:
        return 0.0
    cnt = sum(1 for t in raw_toks if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS))
    return float(round(min(cnt / len(raw_toks), 1.0), 4))

df_features["feat_nltk_opinion_negation_density"] = [calc_feat_nltk_opinion_negation_density(int(r)) for r in df_features["review_id"]]
print(f"feat_nltk_opinion_negation_density -> mean: {df_features['feat_nltk_opinion_negation_density'].mean():.4f}, max: {df_features['feat_nltk_opinion_negation_density'].max():.4f}")

feat_nltk_opinion_negation_density -> mean: 0.1156, max: 0.3889


#### Feature 20: `feat_ebr_aspect_prototype_sim`
Độ tương đồng trung bình Top-3 giữa tập từ khóa sạch của bài đánh giá và từ điển nguyên mẫu (prototype keywords) của 6 khía cạnh trong `[0.0, 1.0]`. Đóng vai trò tín hiệu neo ngữ nghĩa cho cơ chế truy hồi dựa trên embedding (EBR).


In [44]:
def calc_feat_ebr_aspect_prototype_sim(rid):
    clean_toks = review_cache[rid]["clean_tokens"]
    if not clean_toks:
        return 0.0
    tok_counts = Counter(clean_toks)
    sims = []
    for asp in CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_toks) * len(kw_set))
        sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(sims, reverse=True)[:3]
    return float(round(sum(top_sims) / len(top_sims), 4))

df_features["feat_ebr_aspect_prototype_sim"] = [calc_feat_ebr_aspect_prototype_sim(int(r)) for r in df_features["review_id"]]
print(f"feat_ebr_aspect_prototype_sim -> mean: {df_features['feat_ebr_aspect_prototype_sim'].mean():.4f}, min: {df_features['feat_ebr_aspect_prototype_sim'].min():.4f}, max: {df_features['feat_ebr_aspect_prototype_sim'].max():.4f}")

feat_ebr_aspect_prototype_sim -> mean: 0.3029, min: 0.0000, max: 0.7028


In [45]:
FEATURE_COLS_20 = [c for c in df_features.columns if c != "review_id"]
print(f"Đã trích xuất thành công {len(FEATURE_COLS_20)} features cho {len(df_features):,} bài đánh giá:")
print("Danh sách 20 features:", FEATURE_COLS_20)

cleaned_text_pairs = [preprocess_text_paper1(t) for t in df_reviews["review_text"]]
df_reviews["clean_review_text"] = [p[0] for p in cleaned_text_pairs]
df_reviews["paper1_nltk_text"] = [p[1] for p in cleaned_text_pairs]
df_merged = pd.merge(df_reviews, df_features, on="review_id")
display(df_features.head())

Đã trích xuất thành công 20 features cho 9,990 bài đánh giá:
Danh sách 20 features: ['feat_aspect_count', 'feat_dominant_aspect', 'feat_net_sentiment_score', 'feat_sentiment_conflict', 'feat_evidence_density', 'feat_avg_rationale_len', 'feat_has_cross_overlap', 'feat_service_sentiment', 'feat_facility_sentiment', 'feat_experience_sentiment', 'feat_amenity_sentiment', 'feat_loyalty_sentiment', 'feat_branding_sentiment', 'feat_aspect_coverage_ratio', 'feat_neutral_hedging_density', 'feat_consensus_strength', 'feat_lexical_polarity_intensity', 'feat_nltk_clean_token_ratio', 'feat_nltk_opinion_negation_density', 'feat_ebr_aspect_prototype_sim']


,review_id,feat_aspect_count,feat_dominant_aspect,feat_net_sentiment_score,feat_sentiment_conflict,feat_evidence_density,feat_avg_rationale_len,feat_has_cross_overlap,feat_service_sentiment,feat_facility_sentiment,...,feat_amenity_sentiment,feat_loyalty_sentiment,feat_branding_sentiment,feat_aspect_coverage_ratio,feat_neutral_hedging_density,feat_consensus_strength,feat_lexical_polarity_intensity,feat_nltk_clean_token_ratio,feat_nltk_opinion_negation_density,feat_ebr_aspect_prototype_sim
0,1,3,1,0.6,1,0.5572,38.00,0,-0.6,1.0,...,0.0,0.0,0.0,0.5000,0.0149,0.6,-0.6000,0.4328,0.1791,0.4607
1,2,3,3,1.0,0,0.6107,68.57,0,1.0,1.0,...,0.0,0.0,0.0,0.5000,0.0077,1.0,0.7956,0.6462,0.0923,0.3807
2,3,4,3,1.0,0,0.5668,40.00,0,1.0,1.0,...,0.0,1.0,0.0,0.6667,0.0119,1.0,1.0000,0.5595,0.1429,0.2338
3,4,3,1,1.0,0,0.7238,65.50,0,1.0,1.0,...,0.0,1.0,0.0,0.5000,0.0000,1.0,1.0000,0.5052,0.0619,0.3262
4,5,2,2,1.0,0,0.8471,43.20,0,1.0,1.0,...,0.0,0.0,0.0,0.3333,0.0000,1.0,1.0000,0.6154,0.1538,0.3102


### 5.2 Lưu trữ DataFrame thành tệp Parquet và xuất CSDL


In [46]:
os.makedirs("data/df", exist_ok=True)

ann_grouped = df_annotations.groupby("review_id").apply(
    lambda g: g[["annotation_id", "aspect", "sentiment", "start_char", "end_char", "span_text"]].to_dict("records"),
    include_groups=False
).reset_index(name="aspect_annotations")

df_master = pd.merge(df_merged, ann_grouped, on="review_id", how="left")
df_master["aspect_annotations"] = df_master["aspect_annotations"].apply(lambda x: x if isinstance(x, list) else [])

parquet_path = "data/df/hotel_reviews.parquet"
df_master.to_parquet(parquet_path, compression="snappy", index=False)
print(f"Đã xuất tệp Parquet duy nhất của dự án tại: '{parquet_path}' ({len(df_master):,} dòng, {len(df_master.columns)} cột, {len(df_annotations):,} annotations lồng nhau).")

db_path = "data/df/chatbot.db"
df_features.to_sql("review_features", conn, if_exists="replace", index=False)
cursor.execute("CREATE INDEX IF NOT EXISTS idx_feat_review_id ON review_features(review_id);")
conn.commit()
print(f"Đã lưu bảng 'review_features' ({len(df_features):,} dòng) vào SQLite tại '{db_path}'.")

conn.close()
print("Hoàn tất lưu trữ dữ liệu trung gian.")

Đã xuất tệp Parquet duy nhất của dự án tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 38 cột, 54,326 annotations lồng nhau).
Đã lưu bảng 'review_features' (9,990 dòng) vào SQLite tại 'data/df/chatbot.db'.
Hoàn tất lưu trữ dữ liệu trung gian.


### 5.3 Tổng hợp Đa Đánh giá theo Đồng thuận Cảm xúc 3 Lớp (Positive, Neutral, Negative)


In [47]:
def demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity", top_k=5):
    hotel_revs = df_master[df_master["hotel_name"].str.lower() == hotel_target.lower()].copy()
    if hotel_revs.empty:
        hotel_revs = df_master[df_master["hotel_name"].str.lower().str.contains("pullman", na=False)].copy()
    if hotel_revs.empty:
        print(f"Không tìm thấy review cho khách sạn: {hotel_target}")
        return

    actual_hotel_name = hotel_revs.iloc[0]["hotel_name"]

    def mentions_aspect(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return False
        return any(a.get("aspect") == aspect_target for a in ann_list if isinstance(a, dict))

    def get_aspect_sentiment(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return 0
        sents = [a.get("sentiment") for a in ann_list if isinstance(a, dict) and a.get("aspect") == aspect_target]
        if not sents:
            return 0
        pos, neg, neu = sents.count("Positive"), sents.count("Negative"), sents.count("Neutral")
        if pos > neg and pos >= neu:
            return 1
        if neg > pos and neg >= neu:
            return -1
        return 1 if pos > neg else (-1 if neg > pos else 0)

    def get_aspect_spans(ann_list):
        if not hasattr(ann_list, "__iter__"):
            return []
        return [a.get("span_text") for a in ann_list if isinstance(a, dict) and a.get("aspect") == aspect_target and a.get("span_text")]

    hotel_revs["has_aspect"] = hotel_revs["aspect_annotations"].apply(mentions_aspect)
    aspect_subset = hotel_revs[hotel_revs["has_aspect"]].copy()
    if aspect_subset.empty:
        aspect_subset = hotel_revs.copy()

    aspect_subset["aspect_sent_val"] = aspect_subset["aspect_annotations"].apply(get_aspect_sentiment)
    aspect_subset["evidence_spans"] = aspect_subset["aspect_annotations"].apply(get_aspect_spans)
    aspect_subset["evidence_weight"] = aspect_subset["feat_evidence_density"].clip(lower=0.05)

    top_candidates = aspect_subset.sort_values(by=["feat_evidence_density", "feat_consensus_strength"], ascending=False).head(top_k)

    weighted_sent = float(np.average(top_candidates["aspect_sent_val"], weights=top_candidates["evidence_weight"]))
    pos_pct = int(round((top_candidates["aspect_sent_val"] > 0).mean() * 100)) if len(top_candidates) > 0 else 0
    neu_pct = int(round((top_candidates["aspect_sent_val"] == 0).mean() * 100)) if len(top_candidates) > 0 else 0
    neg_pct = max(0, 100 - pos_pct - neu_pct)
    consensus_label = "Positive" if weighted_sent > 0.15 else ("Negative" if weighted_sent < -0.15 else "Neutral")

    print(f"THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP ĐỒNG THUẬN 3 LỚP THEO KHÍA CẠNH: {aspect_target}")
    print(f"Khách sạn: {actual_hotel_name} | Số lượng reviews khớp khía cạnh: {len(top_candidates)} / {len(aspect_subset)}")
    print("Chi tiết các bằng chứng trích xuất theo mật độ dẫn chứng (Evidence Density):")
    for idx, (_, row) in enumerate(top_candidates.iterrows(), 1):
        sent_str = "Positive" if row["aspect_sent_val"] > 0 else ("Negative" if row["aspect_sent_val"] < 0 else "Neutral")
        span_sample = row["evidence_spans"][0] if row["evidence_spans"] else str(row.get("review_text", ""))[:90]
        print(f"  [{idx}] Review #{int(row['review_id'])} | Mật độ bằng chứng: {row['feat_evidence_density']*100:.1f}% | Cảm xúc: {sent_str}")
        print(f"      Dẫn chứng trích đoạn: '{span_sample[:95]}...'")

    print("Tổng hợp đồng thuận cảm xúc 3 lớp:")
    print(f"  - Phân bố 3 lớp: Positive {pos_pct}% | Neutral {neu_pct}% | Negative {neg_pct}%")
    print(f"  - Điểm đồng thuận có trọng số mật độ bằng chứng: {weighted_sent:+.3f} -> Kết luận: {consensus_label}")

demonstrate_multi_review_aspect_synthesis(df_master, hotel_target="Khách sạn pullman hà nội", aspect_target="Amenity")

THỰC NGHIỆM TRÍCH XUẤT ĐA REVIEW VÀ TỔNG HỢP ĐỒNG THUẬN 3 LỚP THEO KHÍA CẠNH: Amenity
Khách sạn: Khách sạn pullman hà nội | Số lượng reviews khớp khía cạnh: 5 / 6
Chi tiết các bằng chứng trích xuất theo mật độ dẫn chứng (Evidence Density):
  [1] Review #777 | Mật độ bằng chứng: 98.1% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Taxis readily available on call and that helps your commute...'
  [2] Review #2177 | Mật độ bằng chứng: 96.7% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'One can walk from Hotel to Ho Chi Minh Mausoleum within 10 minutes, you can also visit the One ...'
  [3] Review #1733 | Mật độ bằng chứng: 78.1% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Club Lounge excellent, good variety of hot and cold food, staff attentive...'
  [4] Review #2721 | Mật độ bằng chứng: 75.4% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Location very convenient...'
  [5] Review #3418 | Mật độ bằng chứng: 52.8% | Cảm xúc: Positive
      Dẫn chứng trích đoạn: 'Checking in

### 5.4 Tiền xử lý Văn bản và Phân chia Tập dữ liệu Huấn luyện


In [48]:
print("Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...")

parquet_path = "data/df/hotel_reviews.parquet"
df_proc = pd.read_parquet(parquet_path)

cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_proc["review_text"]]
df_proc["clean_review_text"] = [p[0] for p in cleaned_pairs]
df_proc["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]

def clean_annotation_spans(anns):
    if not isinstance(anns, (list, np.ndarray)):
        return []
    updated = []
    for ann in anns:
        if isinstance(ann, dict):
            item = dict(ann)
            c_span, nltk_span = preprocess_text_paper1(item.get("span_text", ""))
            item["clean_span_text"] = c_span
            item["paper1_span_tokens"] = nltk_span
            updated.append(item)
    return updated

df_proc["aspect_annotations"] = df_proc["aspect_annotations"].apply(clean_annotation_spans)

sample_paper1 = "I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!"
sample_clean, sample_nltk = preprocess_text_paper1(sample_paper1)
print("Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):")
print("  - Raw Text          :", sample_paper1)
print("  - clean_review_text :", sample_clean)
print("  - paper1_nltk_text  :", sample_nltk)

df_proc["corn_c0"] = (df_proc["sentiment_id"] > 0).astype(int)
df_proc["corn_c1"] = (df_proc["sentiment_id"] > 1).astype(int)

def clean_hotel_name(raw_name: str) -> str:
    if not raw_name:
        return "Hotel"
    text = str(raw_name).replace("Đ", "D").replace("đ", "d")
    norm = unicodedata.normalize("NFKD", text)
    ascii_text = "".join(c for c in norm if not unicodedata.combining(c))
    clean_str = ascii_text.replace("\xa0", " ").strip()
    clean_str = re.sub(r"^(khach\s*san|khu\s*nghi\s*duong|ks|hotel|resort|the)\s+", "", clean_str, flags=re.IGNORECASE).strip()
    clean_str = re.sub(r"\s*\.*$", "", clean_str).strip()
    return clean_str.title() if clean_str else "Hotel"

df_proc["clean_hotel_name"] = df_proc["hotel_name"].apply(clean_hotel_name)

sss_train = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_indices, temp_indices = next(sss_train.split(df_proc, df_proc["sentiment_id"]))

df_temp = df_proc.iloc[temp_indices]
sss_val_test = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_sub_idx, test_sub_idx = next(sss_val_test.split(df_temp, df_proc.iloc[temp_indices]["sentiment_id"]))
val_indices = temp_indices[val_sub_idx]
test_indices = temp_indices[test_sub_idx]

df_proc["split"] = "train"
df_proc.loc[val_indices, "split"] = "val"
df_proc.loc[test_indices, "split"] = "test"

df_proc.to_parquet(parquet_path, compression="snappy", index=False)

db_path = "data/df/chatbot.db"
conn = sqlite3.connect(db_path)
df_proc[["review_id", "clean_review_text", "paper1_nltk_text", "split", "overall_sentiment", "sentiment_id", "corn_c0", "corn_c1"]].to_sql(
    "train_split_info", conn, if_exists="replace", index=False
)
cursor = conn.cursor()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_split ON train_split_info(split);")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sent_id ON train_split_info(sentiment_id);")
conn.commit()
conn.close()

split_counts = df_proc["split"].value_counts()
print(f"\nĐã lưu tệp Parquet hoàn chỉnh tại: '{parquet_path}' ({len(df_proc):,} dòng, {len(df_proc.columns)} cột)")
print("Thống kê phân chia tập dữ liệu:")
for s_name in ["train", "val", "test"]:
    cnt = split_counts.get(s_name, 0)
    pct = (cnt / len(df_proc)) * 100
    print(f"  - Tập {s_name.upper():<5}: {cnt:,} bài đánh giá ({pct:.1f}%)")

print("\nPhân bố tỷ lệ nhãn cảm xúc 3 lớp (% trên từng tập sau khi Stratified Split):")
dist_table = pd.crosstab(df_proc["overall_sentiment"], df_proc["split"], normalize="columns").reindex(["Negative", "Neutral", "Positive"]) * 100
print(dist_table.round(2).to_string())

Bắt đầu tiền xử lý văn bản theo phương pháp paper 1.pdf và đồng bộ tệp hotel_reviews.parquet...
Minh họa tiền xử lý văn bản theo paper 1.pdf (Table 4):
  - Raw Text          : I couldn't imagine a more beautiful place to stay. The staff were amazingly friendly. Could not fault the facilities and service. We didn't want to leave! We will be back for sure and won't stay anywhere else!
  - clean_review_text : i could not imagine a more beautiful place to stay. the staff were amazingly friendly. could not fault the facilities and service. we did not want to leave! we will be back for sure and will not stay anywhere else!
  - paper1_nltk_text  : could not imagine beautiful place stay staff amazingly friendly could not fault facilities service not want leave back sure not stay anywhere else

Đã lưu tệp Parquet hoàn chỉnh tại: 'data/df/hotel_reviews.parquet' (9,990 dòng, 42 cột)
Thống kê phân chia tập dữ liệu:
  - Tập TRAIN: 7,992 bài đánh giá (80.0%)
  - Tập VAL  : 999 bài đánh giá (10.0%)
 